**Paper:** *A Strategic Engineering Analysis of Flexible and Robust Deployment of Solar PV Power Plants Coupled with Battery Energy Storage Systems*

**Authors:** João Graça Gomes, Michel-Alexandre Cardin, Billy Wu — Dyson School of Design Engineering, Imperial College London

---

### Overview

This notebook implements the three-step methodology described in the paper:

| Step | Model Type | Description | Paper Reference |
|------|-----------|-------------|-----------------|
| **Step 1** | Deterministic | Baseline LCOE/LCOS for fixed & phased PV+BESS deployment schedules | Eqs. (1)–(16) |
| **Step 2** | Stochastic | Monte Carlo simulation with uncertain costs and curtailment (EA) | Eqs. (17)–(20) |
| **Step 3** | Flexible (ROA) | Adaptive capacity expansion via EA-triggered decision rules | Eqs. (21)–(30) |

The core idea follows the Strategic Engineering framework (Cardin, 2014):
rather than optimising for a single forecast, the model evaluates **flexible deployment strategies**
that can adapt as uncertainty unfolds—quantifying the **Value of Flexibility (VoF)** through
distributional LCOE comparisons across fixed, phased, and adaptive configurations.

### Case Study
- **Location:** Alentejo, Portugal (high solar irradiance region)
- **System:** 100 MW Solar PV + 60 MW / 60 MWh LFP BESS
- **Horizon:** 20-year operational lifetime
- **Uncertainty:** Effective Availability (EA), technology costs, O&M costs


## 1. Library Imports

Standard scientific Python stack for numerical simulation, statistical analysis,
and publication-quality visualisation.


In [ ]:
# =============================================================================
# LIBRARY IMPORTS
# =============================================================================
# Core numerical and data-handling libraries
import pandas as pd                         # Tabular data manipulation
import numpy as np                          # Numerical arrays and linear algebra
import matplotlib.pyplot as plt             # Plotting engine
import matplotlib.image as mpimg            # Image display in figures
import scipy.stats as stats                 # Statistical distributions
import seaborn as sns                       # Statistical data visualisation

# Specialised utilities
from scipy.interpolate import interp1d     # 1-D interpolation (for EA curves)
from scipy.optimize import minimize         # Optimisation (reserved for extensions)
from scipy.stats import gaussian_kde        # Kernel density estimation
from matplotlib.lines import Line2D         # Custom legend handles


## 2. Project-Level Constants and Techno-Economic Parameters

These constants define the **baseline case study** described in Section 3 of the paper.

| Parameter | Value | Meaning |
|-----------|-------|---------|
| `Power_capacity` | 100 MW | Nameplate PV plant capacity |
| `Dolar` | 0.9243 | USD → EUR conversion factor |
| `project_lifetime` | 20 years | Operational horizon *T* |
| `discount_rate` (λ) | 3 % | Nominal discount rate — Eq. (1) |
| `degradation_rate` (∂) | 0.52 %/yr | Annual PV module degradation |
| `Learning_rate_technology` | 20 % | Exogenous learning rate LR_exo — Eq. (16) |
| `Learning_rate_experience` | 3 % | Endogenous learning rate LR_end — Eq. (15) |


In [ ]:
# =============================================================================
# PROJECT-LEVEL CONSTANTS
# =============================================================================

Power_capacity = 100          # [MW] Nameplate PV plant capacity (θ_max in paper)
Dolar = 0.9243                # USD-to-EUR conversion factor (all BESS cost data is in USD)
project_lifetime = 20         # [years] Planning horizon T — Eq. (1)
discount_rate = 0.03          # [-] Nominal discount rate λ — Eq. (1)

# --- Sensitivity sweep vectors (used in Step 3 / sensitivity analysis) ---
# discount_rates3A = [0.01, 0.03, 0.05, 0.07, 0.09, 0.1]
Economies_scale1 = [1]                              # EoS exponent α = 1 → no scale advantage (baseline)
Economies_scale3A = [0.6, 0.7, 0.8, 0.9, 1]        # Sweep over α — Eq. (12)

# --- Solar PV technical parameters ---
degradation_rate = 0.0052     # [-] Annual PV module degradation ∂ — Eq. (9)
degradation_rates3A = [0.0052, 0.01, 0.03, 0.05, 0.07, 0.09]  # Sensitivity sweep

Conversion_power = 10         # Scaling factor: raw solar data → MW-scale generation
Central_EA = 0.85             # [-] Central Effective Availability (1 - curtailment fraction)

# --- Learning rates — Eqs. (14)–(18) ---
Learning_rate_technology = 0.2    # LR_exo: 20% cost reduction per doubling of global capacity
Learning_rate_technology3A = [0.05, 0.1, 0.15, 0.2, 0.3, 0.4]  # Sensitivity sweep
Learning_rate_experience = 0.03   # LR_end: 3% cost reduction per doubling of local capacity

learning_factor_module = 1 - Learning_rate_technology   # Multiplicative factor for modules
learning_overall = 1 - Learning_rate_experience         # Multiplicative factor for local learning

# --- Other PV costs (site-specific, Portugal) ---
Feasibility_studies_engineering_design = 35000  # [€/MW] Engineering design & preliminary studies
Land_Renting = 2342                             # [€/MW] Annual land rental cost — PV_area(θ)·L_cost
Salvage = 28000 * Dolar                         # [€/MW] End-of-life decommissioning — C_dec,PV

# --- BESS learning rate ---
learning_rate_BESS1 = 0.1                               # 10% baseline BESS learning rate
learning_rate_BESS3A = [0.02, 0.05, 0.1, 0.15, 0.2, 0.3]  # Sensitivity sweep


## 3. PV Module, Tracking System, and Operational Cost Data

The model evaluates six PV module technologies and three mounting/tracking configurations.
Costs are sourced from industry benchmarks and converted to EUR.

**Module technologies** span the efficiency frontier described in Section 3:
Al-BSF (baseline), PERC, Multi-PERC, Bifacial PERT, Bifacial SHJ, and IBC (highest efficiency, +12%).

**Tracker types** affect both CAPEX and annual energy yield via the multiplier *A_trac* — Eq. (8).
OpEx varies significantly: dual-axis systems cost ~6× more to maintain than fixed-tilt.


In [ ]:
# =============================================================================
# PV MODULE COST DATA — C_mod in Eq. (5)
# =============================================================================
# Cost per watt for each module technology, converted to EUR
module_data = {
    "Standard_AI_BSF": {"cost_per_watt": 0.39 * Dolar},   # Aluminium Back Surface Field
    "PERC":            {"cost_per_watt": 0.38 * Dolar},   # Passivated Emitter and Rear Cell
    "Multi_perc":      {"cost_per_watt": 0.37 * Dolar},   # Multi-crystalline PERC
    "Bifacial_pert":   {"cost_per_watt": 0.40 * Dolar},   # Bifacial PERT
    "Bifacial_SHJ":    {"cost_per_watt": 0.40 * Dolar},   # Silicon Heterojunction (bifacial)
    "IBC":             {"cost_per_watt": 0.41 * Dolar},   # Interdigitated Back Contact (highest η)
}

# =============================================================================
# TRACKING SYSTEM COSTS — C_trac in Eq. (5)
# =============================================================================
# Costs per watt, tiered by installed capacity bracket (economies of scale)
tracking_costs = {
    "Fixed": {
        "below5MW":   0.70 * Dolar,
        "below10MW":  0.60 * Dolar,
        "below50MW":  0.49 * Dolar,
        "below100MW": 0.58 * Dolar,
    },
    "OneAxis": {
        "below5MW":   (1.22 - 0.34) * Dolar,    # Single-axis tracker premium
        "below10MW":  (1.13 - 0.34) * Dolar,
        "below50MW":  (0.98 - 0.34) * Dolar,
        "below100MW": (0.89 - 0.34) * Dolar,
    },
    "DualAxis": {
        "below5MW":   (1.13 + 1.015 - 0.34) * Dolar,   # Dual-axis: highest yield, highest cost
        "below10MW":  (1.03 + 1.015 - 0.34) * Dolar,
        "below50MW":  (0.91 + 1.015 - 0.34) * Dolar,
        "below100MW": (0.83 + 1.015 - 0.34) * Dolar,
    },
}

# =============================================================================
# MODULE EFFICIENCY MULTIPLIERS — η_mod in Eq. (8)
# =============================================================================
# Relative to Al-BSF baseline (1.0)
module_efficiency = {
    "Standard_AI_BSF": 1.00,    # Baseline
    "PERC":            1.03,    # +3% energy yield
    "Multi_perc":      1.02,    # +2%
    "Bifacial_pert":   1.05,    # +5%
    "Bifacial_SHJ":    1.10,    # +10%
    "IBC":             1.12,    # +12% (best-in-class)
}

# =============================================================================
# TRACKER ENERGY YIELD MULTIPLIERS — A_trac in Eq. (8)
# =============================================================================
tracker_multipliers = {
    "Fixed":    1.000,   # Baseline: fixed-tilt mounting
    "OneAxis":  1.096,   # +9.6% annual yield from single-axis tracking
    "DualAxis": 1.395,   # +39.5% annual yield from dual-axis tracking
}

# =============================================================================
# PV OPERATIONAL EXPENDITURE (OpEx) — OM_fix,PV in Eq. (5)
# =============================================================================
# Annual fixed O&M cost per kW installed, by tracker type
opex_per_kw_per_year = {
    "Fixed":    17,     # €17/kW/yr
    "OneAxis":  36,     # €36/kW/yr
    "DualAxis": 101,    # €101/kW/yr — reflects mechanical maintenance burden
}

#DATA SOURCES MENTIONED IN THE ANNEX 1 OF THE PAPER.

## 4. Battery Energy Storage System (BESS) Cost Data

BESS costs are loaded from an external Excel workbook containing LFP (Lithium Iron Phosphate)
cost breakdowns at three capacity tiers (1 MW, 10 MW, 100 MW), each with multiple storage
durations (1–10 hours). This implements the cost structure in **Eq. (6)** of the paper.

Cost components map directly to the paper's notation:
- **DC Storage Block** → DC_C (cell cost, €/kWh)
- **DC Storage BOS** → DC_BOS (cabling, racks, protection, €/kWh)
- **Systems Integration** → SI (installation & commissioning, €/kWh)
- **Power Equipment** → Peq (inverters, power electronics, €/kW)
- **CC** → CC (battery container cost, €/kW)
- **Grid Integration** → G_int (interconnection, transformers, €/kW)

> **Note:** The Excel file path must be updated to your local environment.


In [ ]:
# =============================================================================
# BESS TECHNICAL AND COST PARAMETERS
# =============================================================================

# Path to LFP battery cost workbook (update for your environment)
Battery_data = r"/Battery/Cost_battery.xlsx" #SOURCE: Pacific Northwestern Laboratory, https://www.pnnl.gov/projects/esgc-cost-performance

grid_ratio = 0.2            # [-] Fraction of grid electricity used for BESS arbitrage
battery_efficiency = 0.83   # [-] Round-trip efficiency η_char × η_dis — Eqs. (10)–(11)
DoD = 0.8                   # [-] Depth of Discharge (usable fraction of nameplate capacity)
Battery_degradation1 = 0.001  # [-] Annual BESS degradation rate ∂ — Eq. (9)
Battery_degradation3 = [0.0052, 0.01, 0.03, 0.05, 0.07, 0.09]  # Sensitivity sweep
Salvage_battery = 2.39      # [€/kWh] End-of-life salvage value — C_dec,BESS

# =============================================================================
# LOAD BESS COST DATA FROM EXCEL — three capacity tiers
# =============================================================================

# --- 1 MW tier ---
Cost_1MW_Battery_data_df = pd.read_excel(
    Battery_data, sheet_name="LFB", usecols="A:K", header=0, index_col=0, nrows=10
)
Cost_1MW_Battery_data_df = Cost_1MW_Battery_data_df.apply(pd.to_numeric, errors="coerce")
cost_dict = Cost_1MW_Battery_data_df.to_dict()  # Small-scale BESS cost lookup

# Verify data loaded correctly
print("Available Storage Durations:", list(cost_dict.keys()))
cost_4h = cost_dict["4 hours"]["DC Storage Block ($/kWh)"]
print(f"Cost of DC Storage Block for 4 hours: {cost_4h}")
VariableD = cost_dict["6 hours"]["DC Storage Block ($/kWh)"] * 2
print(f"VariableD: {VariableD}")

# --- 10 MW tier ---
Cost_10MW_Battery_data_df = pd.read_excel(
    Battery_data, sheet_name="LFB", usecols="A:K", header=11, index_col=0, nrows=10
)
Cost_10MW_Battery_data_df = Cost_10MW_Battery_data_df.apply(pd.to_numeric, errors="coerce")
cost_dict_10 = Cost_10MW_Battery_data_df.to_dict()  # Medium-scale BESS cost lookup

cost_6h = cost_dict_10["6 hours"]["DC Storage Block ($/kWh)"]
print(f"Cost of DC Storage Block for 6 hours: {cost_6h}")

# --- 100 MW tier ---
Cost_100MW_Battery_data_df = pd.read_excel(
    Battery_data, sheet_name="LFB", usecols="A:K", header=22, index_col=0, nrows=10
)
Cost_100MW_Battery_data_df = Cost_100MW_Battery_data_df.apply(pd.to_numeric, errors="coerce")
cost_dict_100 = Cost_100MW_Battery_data_df.to_dict()  # Utility-scale BESS cost lookup

cost_6h = cost_dict_100["6 hours"]["DC Storage Block ($/kWh)"]
print(f"Cost of DC Storage Block for 6 hours: {cost_6h}")


## 5. Solar Irradiation Data Loading

Hourly solar power generation profiles for the Alentejo, Portugal site are loaded
from historical data (2014–2023). These provide the physical basis for **PV_gen(θ)**
in Eq. (8). Each profile contains 8760 hourly power values, scaled by `Conversion_power` to
represent output at MW-scale.

> **Note:** File paths must be updated to your local environment.


In [ ]:
# =============================================================================
# SOLAR IRRADIATION DATA — Historical hourly generation profiles (2014-2023)
# =============================================================================
# Data source: Alentejo, Portugal site — provides GHI and temperature-adjusted
# power output used to compute PV_gen(θ) — Eq. (8)
#
# NOTE: Update file paths to your local environment.

solar2014_file = r"/Solarpower_2014.xls"
solar2015_file = r"/Solarpower_2015.xls"
solar2016_file = r"/Solarpower_2016.xls"
solar2017_file = r"/Solarpower_2017.xls"
solar2018_file = r"/Solarpower_2018.xls"
solar2019_file = r"/Solarpower_2019.xls"
solar2020_file = r"/Solarpower_2020.xls"
solar2021_file = r"/Solarpower_2021.xls"
solar2022_file = r"/Solarpower_2022.xls"
solar2023_file = r"/Solarpower_2023.xls"


solarpower2014_df = pd.read_excel(solar2014_file, sheet_name='Folha1', usecols='B:D', nrows=8759)
solarpower2015_df = pd.read_excel(solar2015_file, sheet_name='Folha1', usecols='B:D', nrows=8759)
solarpower2016_df = pd.read_excel(solar2016_file, sheet_name='Folha1', usecols='B:D', nrows=8759)
solarpower2017_df = pd.read_excel(solar2017_file, sheet_name='Folha1', usecols='B:D', nrows=8759)
solarpower2018_df = pd.read_excel(solar2018_file, sheet_name='Folha1', usecols='B:D', nrows=8759)
solarpower2019_df = pd.read_excel(solar2019_file, sheet_name='Folha1', usecols='B:D', nrows=8759)
solarpower2020_df = pd.read_excel(solar2020_file, sheet_name='Folha1', usecols='B:D', nrows=8759)
solarpower2021_df = pd.read_excel(solar2021_file, sheet_name='Folha1', usecols='B:D', nrows=8759)
solarpower2022_df = pd.read_excel(solar2022_file, sheet_name='Folha1', usecols='B:D', nrows=8759)
solarpower2023_df = pd.read_excel(solar2023_file, sheet_name='Folha1', usecols='B:D', nrows=8759)

solarpower2014_df['HourIndex'] = solarpower2014_df['Day'] * 24 + solarpower2014_df['Hour']
P2014 = [Conversion_power*power for power in solarpower2014_df.set_index('HourIndex')['Power'].reindex(range(1, 8761), fill_value=0).tolist()]
#print("Solar Generation 2014",sum(P2014))
solarpower2015_df['HourIndex'] = solarpower2015_df['Day'] * 24 + solarpower2015_df['Hour']
P2015 = [Conversion_power*power for power in solarpower2015_df.set_index('HourIndex')['Power'].reindex(range(1, 8761), fill_value=0).tolist()]
#print("Solar Generation 2015",sum(P2015))
solarpower2016_df['HourIndex'] = solarpower2016_df['Day'] * 24 + solarpower2016_df['Hour']
P2016 = [Conversion_power*power for power in solarpower2016_df.set_index('HourIndex')['Power'].reindex(range(1, 8761), fill_value=0).tolist()]
#print("Solar Generation 2016",sum(P2016))
solarpower2017_df['HourIndex'] = solarpower2017_df['Day'] * 24 + solarpower2017_df['Hour']
P2017 = [Conversion_power*power for power in solarpower2017_df.set_index('HourIndex')['Power'].reindex(range(1, 8761), fill_value=0).tolist()]
#print("Solar Generation 2017",sum(P2017))
solarpower2018_df['HourIndex'] = solarpower2018_df['Day'] * 24 + solarpower2018_df['Hour']
P2018 = [Conversion_power*power for power in solarpower2018_df.set_index('HourIndex')['Power'].reindex(range(1, 8761), fill_value=0).tolist()]
#print("Solar Generation 2018",sum(P2018))
solarpower2019_df['HourIndex'] = solarpower2019_df['Day'] * 24 + solarpower2019_df['Hour']
P2019 = [Conversion_power*power for power in solarpower2019_df.set_index('HourIndex')['Power'].reindex(range(1, 8761), fill_value=0).tolist()]
#print("Solar Generation 2019",sum(P2019))
solarpower2020_df['HourIndex'] = solarpower2020_df['Day'] * 24 + solarpower2020_df['Hour']
P2020 = [Conversion_power*power for power in solarpower2020_df.set_index('HourIndex')['Power'].reindex(range(1, 8761), fill_value=0).tolist()]
#print("Solar Generation 2020",sum(P2020))
solarpower2021_df['HourIndex'] = solarpower2021_df['Day'] * 24 + solarpower2021_df['Hour']
P2021 = [Conversion_power*power for power in solarpower2021_df.set_index('HourIndex')['Power'].reindex(range(1, 8761), fill_value=0).tolist()]
#print("Solar Generation 2021",sum(P2021))
solarpower2022_df['HourIndex'] = solarpower2022_df['Day'] * 24 + solarpower2022_df['Hour']
P2022 = [Conversion_power*power for power in solarpower2022_df.set_index('HourIndex')['Power'].reindex(range(1, 8761), fill_value=0).tolist()]
#print("Solar Generation 2022",sum(P2022))
solarpower2023_df['HourIndex'] = solarpower2023_df['Day'] * 24 + solarpower2023_df['Hour']
P2023 = [Conversion_power*power for power in solarpower2023_df.set_index('HourIndex')['Power'].reindex(range(1, 8761), fill_value=0).tolist()]
#print("Solar Generation 2023",sum(P2023))
###random power production sets
Power_list = [P2014, P2015, P2016, P2017, P2018, P2019, P2020, P2021]  
power_array = np.array(Power_list)
average_power = np.mean(power_array, axis=0)

# -------------------------------------------------------------------------
# Assign hourly generation profiles to each year of the planning horizon.
#   P1–P10  : Historical profiles (2014–2023), used directly.
#   P11–P20 : Synthetic profiles for the projection period (2024–2033),
#             constructed above from the historical average with
#  P21–P24 : Extended beyond the primary horizon using the last
#             available synthetic profile, to cover the residual
#             operating years of capacity installed in later phases.
# -------------------------------------------------------------------------

P1 = np.array(P2014)
P2 = np.array(P2015)
P3 = np.array(P2016)
P4 = np.array(P2017)
P5 = np.array(P2018)
P6 = np.array(P2019)
P7 = np.array(P2020)
P8 = np.array(P2021)
P9 = np.array(P2022)
P10 = np.array(P2023)
P11 = np.array(P2024)
P12 = np.array(P2025)
P13 = np.array(P2026)
P14 = np.array(P2027)
P15 = np.array(P2028)
P16 = np.array(P2029)
P17 = np.array(P2030)
P18 = np.array(P2031)
P19 = np.array(P2032)
P20 = np.array(P2033)
P21 = np.array(P2033)
P22 = np.array(P2033)
P23 = np.array(P2033)
P24 = np.array(P2033)

P_profiles = [
    np.array(P2014).sum(), np.array(P2015).sum(), np.array(P2016).sum(), np.array(P2017).sum(),
    np.array(P2018).sum(), np.array(P2019).sum(), np.array(P2020).sum(), np.array(P2021).sum(),
    np.array(P2022).sum(), np.array(P2023).sum(), np.array(P2024).sum(), np.array(P2025).sum(),
    np.array(P2026).sum(), np.array(P2027).sum(), np.array(P2028).sum(), np.array(P2029).sum(),
    np.array(P2030).sum(), np.array(P2031).sum(), np.array(P2032).sum(), np.array(P2033).sum()
]

Total_generation=sum(P_profiles)
print(f"Total generation over lifetime {Total_generation} MWh")


## 6. Deployment Scenarios and Cost Category Helper

### PV Installation Scenarios
Five **predetermined deployment schedules** for the 100 MW PV plant, ranging from
a single upfront commitment ("Fixed" in the paper) to phased multi-year rollouts.
These map to the capacity vector **χ** in Eq. (24).

### BESS Installation Scenarios
Five deployment schedules for the 60 MW BESS, similarly ranging from fixed to phased.

### Cost Category Function
Determines the BESS cost tier based on the capacity being installed in a given year.
Smaller tranches pay higher per-unit costs (no economies of scale).


In [ ]:
# =============================================================================
# PV DEPLOYMENT SCHEDULES — χ = (θ₀, θ₁, ..., θ_T)
# =============================================================================
# Each list: [MW installed in Year 0, Year 1, ..., Year T]
# Sum always equals 100 MW (θ_max); only the timing differs.
installation_scenarios = {
    "100 MW in Year 0":                        [100] + [0] * (project_lifetime),      # Fixed: full upfront
    "50 MW in Year 0 and 50 MW in Year 1":     [50, 50] + [0] * (project_lifetime - 1),  # 2-phase
    "50 MW first year, 25 MW next two years":  [50, 25, 25] + [0] * (project_lifetime - 2),  # 3-phase
    "75 MW first year, 25 MW next year":       [75, 25] + [0] * (project_lifetime - 1),  # Front-loaded 2-phase
    "25 MW first year, 25 MW next years":      [25, 25, 25, 25] + [0] * (project_lifetime - 3),  # 4-phase (most modular)
}

# =============================================================================
# COST CATEGORY HELPER — selects BESS cost tier based on capacity
# =============================================================================
def determine_cost_category(installed_capacity_mw):
    """
    Map installed capacity (MW) to a cost bracket for tracker and BESS pricing.
    Reflects economies of scale: larger installations get lower per-unit costs.
    """
    if installed_capacity_mw < 6:
        return "below5MW"
    elif installed_capacity_mw < 11:
        return "below10MW"
    elif installed_capacity_mw < 51:
        return "below50MW"
    else:
        return "below100MW"

# =============================================================================
# BESS DEPLOYMENT SCHEDULES — (δ₀, δ₁, ..., δ_T)
# =============================================================================
# Storage duration fixed at 1 hour for the base case (1 MWh per MW of BESS)
storage_durations2 = [1]  # [hours] — S_hours in Eq. (6)

battery_installation_scenarios = {
    "60 MW in Year 0":                          [60] + [0] * (project_lifetime),           # Fixed: full upfront
    "50 MW in Year 0 and 10 MW in Year 1":      [50, 10] + [0] * (project_lifetime - 1),   # 2-phase
    "20 MW first year, 20 MW next 2 years":     [20, 20, 20] + [0] * (project_lifetime - 2),  # 3-phase (equal)
    "30 MW first year, 30 MW next 1 year":      [30, 30] + [0] * (project_lifetime - 1),   # 2-phase (equal)
    "10 MW first year, 20 MW next 1 year, 30":  [10, 20, 30] + [0] * (project_lifetime - 2),  # 3-phase (escalating)
}


---
## Step 1 — Deterministic LCOE/LCOS Evaluation

**Paper reference: Section 2.1.1, Eqs. (1)–(19)**

This section implements the **deterministic model** that evaluates all combinations of:
- 5 PV deployment schedules × 5 BESS deployment schedules
- 6 module technologies × 3 tracker types
- = **450 configurations** total

For each configuration, the model computes:

1. **LCOE** — Eq. (1): Σ discounted costs / Σ discounted electricity delivered
2. **LCOS** — Eq. (19): Σ discounted BESS costs / Σ discounted BESS discharge

Key modelling features:
- **Learning curves** reduce module costs (LR_exo = 20%) and tracker costs (LR_end = 3%)
  for capacity installed in later years — Eqs. (14)–(18)
- **PV degradation** at 0.52%/year reduces annual generation — Eq. (9)
- **BESS degradation** at 0.1%/year reduces usable capacity
- **Decommissioning** after 20 years of operation with salvage costs
- **Battery cycling**: 4500 cycles / 20 years = 225 cycles/year

This step establishes the **baseline cost landscape** against which flexible
strategies (Step 3) will be compared to compute the Value of Flexibility.


In [ ]:
# =============================================================================
# STEP 1: DETERMINISTIC LCOE / LCOS COMPUTATION
# =============================================================================
# Exhaustive evaluation of all (PV scenario × BESS scenario × module × tracker)
# combinations under deterministic assumptions (no uncertainty).
#
# This implements the deterministic model of Section 2.1.1:
#   LCOE = Σ C_t(θ,δ) / (1+λ)^t  ÷  Σ E_t / (1+λ)^t   — Eq. (1)
#   LCOS = Σ C_t^BESS(δ) / (1+λ)^t  ÷  Σ η_dis·B_dis / (1+λ)^t  — Eq. (19)

results2 = []

for scenario_name_battery, installation_schedule_battery in battery_installation_scenarios.items():    
    total_battery_capacity_mw = sum(installation_schedule_battery)
    
    for storage_hours in storage_durations2:      
        storage_hours_rounded = int(round(storage_hours))
        
        for scenario_name, installation_schedule in installation_scenarios.items():
            for tracker, tracker_cost_dict in tracking_costs.items():
                tracker_multiplier = tracker_multipliers[tracker]       # A_trac — Eq. (8)
                tracker_opex_per_kw = opex_per_kw_per_year[tracker]    # OM_fix,PV

                for module, module_info in module_data.items():
                    # Determine when the last capacity tranche is installed
                    last_installation_year = max(idx for idx, cap in enumerate(installation_schedule) if cap > 0)
                    extended_project_lifetime = last_installation_year + 21  # Each tranche operates for 20 years

                    module_efficiency_multiplier = module_efficiency[module]  # η_mod — Eq. (8)

                    # --- Initialise financial accumulators ---
                    total_solar_capex = 0.0                # Σ C_PV(θ) — Eq. (5)
                    total_battery_capex = 0.0              # Σ C_BESS(δ) — Eq. (6)
                    npv_numerator_lcoe = 0.0               # Σ OpEx (PV + BESS)
                    npv_denominator_lcoe = 0.0             # Σ E_t / (1+λ)^t — Eq. (7)
                    npv_numerator_lcos = 0.0               # Σ BESS OpEx only
                    npv_denominator_lcos = 0.0             # Σ BESS discharge / (1+λ)^t
                    cumulative_installed_capacity_mw = 0    # Running total θ
                    cumulative_battery_installed_capacity_mw = 0  # Running total δ
                    active_generations = [0] * extended_project_lifetime
                    battery_generations_discharge = [0] * extended_project_lifetime
                    Salvage_total_lcoe = 0                 # Accumulated decommissioning costs (PV)
                    Salvage_total_lcos = 0                 # Accumulated decommissioning costs (BESS)
                    decommissioned_capacity = 0
                    decommissioned_capacity_battery = 0

                    # --- Year-by-year simulation over the extended lifetime ---
                    for year in range(extended_project_lifetime):
                        discount_factor = 1 / ((1 + discount_rate) ** year) if year > 0 else 1

                        # Capacity installed THIS year
                        installed_capacity_mw = installation_schedule[year] if year < len(installation_schedule) else 0
                        battery_installed_capacity_mw = installation_schedule_battery[year] if year < len(installation_schedule_battery) else 0

                        # Decommission capacity that has reached its 20-year lifetime
                        if year >= 20:
                            decommissioned_capacity = installation_schedule[year - 20]
                            cumulative_installed_capacity_mw -= decommissioned_capacity
                            decommissioned_capacity_battery = installation_schedule_battery[year - 20]
                            cumulative_battery_installed_capacity_mw -= decommissioned_capacity_battery

                        # Update cumulative installed capacities
                        cumulative_installed_capacity_mw += installed_capacity_mw
                        cumulative_battery_installed_capacity_mw += battery_installed_capacity_mw

                        # --- PV CAPEX calculation — Eq. (5) with learning — Eq. (17) ---
                        if installed_capacity_mw > 0:
                            cost_category = determine_cost_category(installed_capacity_mw)
                            tracker_cost_per_watt = tracker_cost_dict[cost_category]

                            # Learning factors: cost reduction for later-year installations
                            lf_tech = (1 - Learning_rate_technology) ** max(0, year - 1)   # Exogenous — Eq. (16)
                            lf_exp = (1 - Learning_rate_experience) ** max(0, year - 1)    # Endogenous — Eq. (15)

                            plant_capacity_w = installed_capacity_mw * 1e6  # Convert MW → W
                            module_cost = plant_capacity_w * module_info["cost_per_watt"] * lf_tech
                            tracker_cost = plant_capacity_w * tracker_cost_per_watt * lf_exp
                            yearly_solar_capex = module_cost + tracker_cost
                            total_solar_capex += yearly_solar_capex * discount_factor

                        # --- BESS CAPEX calculation — Eq. (6) with learning — Eq. (18) ---
                        # Select cost tier based on installed BESS capacity this year
                        if battery_installed_capacity_mw < 10:
                            battery_cost_dict = cost_dict          # 1 MW tier
                        elif battery_installed_capacity_mw < 61:
                            battery_cost_dict = cost_dict_10       # 10 MW tier
                        else:
                            battery_cost_dict = cost_dict_100      # 100 MW tier

                        lf_bess = (1 - learning_rate_BESS1) ** max(0, year - 1)  # BESS learning factor

                        battery_capacity_kwh = battery_installed_capacity_mw * 1e3 * storage_hours_rounded
                        battery_capacity_kw = battery_installed_capacity_mw * 1e3

                        # Total BESS cost = energy-proportional costs (€/kWh) + power-proportional costs (€/kW)
                        battery_cost = (
                            battery_capacity_kwh * sum(
                                battery_cost_dict[f"{storage_hours_rounded} hours"][k] 
                                for k in ["DC Storage Block ($/kWh)", "DC Storage BOS ($/kWh)", 
                                          "Systems Integration ($/kWh)", "EPC ($/kWh)", 
                                          "Project Development ($/kWh)"]
                            ) * Dolar * lf_bess +
                            battery_capacity_kw * sum(
                                battery_cost_dict[f"{storage_hours_rounded} hours"][k] 
                                for k in ["Power Equipment ($/kW)", "CC ($/kW)", 
                                          "Grid Integration ($/kW)"]
                            ) * Dolar * lf_bess
                        )
                        total_battery_capex += battery_cost * discount_factor

                        # --- Energy generation — Eqs. (7)–(8) ---
                        active_generation_for_year = 0
                        battery_generation_for_year = 0

                        if year > 0:
                            # PV generation: sum contributions from all previously installed tranches
                            for previous_year in range(max(0, year - 19), year + 1):
                                active_capacity_mw = installation_schedule[previous_year] if previous_year < len(installation_schedule) else 0
                                degradation_factor = (1 - degradation_rate) ** (year - previous_year)  # Eq. (9)
                                active_generation_for_year += (
                                    P_profiles[min(year, len(P_profiles) - 1)]    # Annual solar resource
                                    * active_capacity_mw / Power_capacity          # Scale to tranche size
                                    * tracker_multiplier                           # A_trac — Eq. (8)
                                    * module_efficiency_multiplier                 # η_mod — Eq. (8)
                                    * degradation_factor                           # (1-∂)^t — Eq. (9)
                                )

                            # BESS discharge: annual energy from battery arbitrage
                            cycles_per_year = 4500 / 20  # 225 cycles/year over 20-year battery life
                            for previous_year in range(max(0, year - 19), year + 1):
                                battery_active_capacity_mw = installation_schedule_battery[previous_year] if previous_year < len(installation_schedule_battery) else 0
                                degradation_battery_factor = (1 - Battery_degradation1) ** (year - previous_year)
                                battery_generation_for_year += (
                                    battery_active_capacity_mw
                                    * storage_hours_rounded       # S_hours — Eq. (9)
                                    * battery_efficiency           # η_dis — Eq. (7)
                                    * DoD                          # Depth of Discharge
                                    * degradation_battery_factor   # Battery degradation
                                    * cycles_per_year              # Annual cycling
                                )

                            # Accumulate discounted OpEx (numerator) and generation (denominator)
                            npv_numerator_lcoe += (
                                tracker_opex_per_kw * cumulative_installed_capacity_mw * 1e3 * discount_factor +
                                Land_Renting / 20 * installed_capacity_mw * discount_factor +
                                battery_cost_dict[f"{storage_hours_rounded} hours"]["Operation Cost ($/kW-year)"] * battery_capacity_kw * discount_factor
                            )
                            npv_numerator_lcos += (
                                battery_cost_dict[f"{storage_hours_rounded} hours"]["Operation Cost ($/kW-year)"] * battery_capacity_kw * discount_factor
                            )

                            npv_denominator_lcoe += (active_generation_for_year + battery_generation_for_year) * discount_factor
                            npv_denominator_lcos += battery_generation_for_year * discount_factor

                        active_generations[year] = active_generation_for_year
                        battery_generations_discharge[year] = battery_generation_for_year

                        # Salvage/decommissioning costs
                        if decommissioned_capacity > 0:
                            salvage_year = Salvage * decommissioned_capacity * discount_factor
                            Salvage_total_lcoe += salvage_year
                        if decommissioned_capacity_battery > 0:
                            salvage_year_battery = Salvage_battery * 1e3 * storage_hours_rounded * decommissioned_capacity_battery * discount_factor
                            Salvage_total_lcos += salvage_year_battery

                    # --- Final LCOE and LCOS — Eqs. (1) and (19) ---
                    salvage_total = Salvage_total_lcoe + Salvage_total_lcos
                    salvage_total_battery = Salvage_total_lcos

                    lcoe = (total_solar_capex + total_battery_capex + npv_numerator_lcoe + salvage_total) / npv_denominator_lcoe if npv_denominator_lcoe != 0 else float("inf")
                    lcos = (total_battery_capex + npv_numerator_lcos + salvage_total_battery) / npv_denominator_lcos if npv_denominator_lcos != 0 else float("inf")

                    results2.append({
                        "Scenario PV": scenario_name,
                        "Scenario Battery": scenario_name_battery,
                        "Tracker Type": tracker,
                        "Module Type": module,
                        "Battery Capacity (MW)": total_battery_capacity_mw,
                        "Storage Hours": storage_hours_rounded,
                        "Solar CAPEX (EUR)": total_solar_capex,
                        "Battery CAPEX (EUR)": total_battery_capex,
                        "Total CAPEX (EUR)": total_solar_capex + total_battery_capex,
                        "LCOE (EUR/MWh)": lcoe,
                        "LCOS (EUR/MWh)": lcos,
                        "Total Solar Capacity (MW)": sum(installation_schedule),
                        "Total Battery Capacity (MWh)": total_battery_capacity_mw * storage_hours_rounded
                    })

results2_df = pd.DataFrame(results2)
print(f"Step 1 complete: {len(results2_df)} configurations evaluated")


### Step 1 — Results Summary and Visualisation

Heatmaps show the deterministic LCOE landscape across all PV × BESS deployment combinations.
Darker colours indicate lower (better) LCOE. The analysis filters down to **Fixed tracker + IBC module**
as the optimal technology combination for subsequent steps.


In [ ]:
# =============================================================================
# STEP 1 RESULTS — Summary statistics and heatmap visualisations
# =============================================================================
# Computes descriptive statistics (mean, min, max) for LCOE and LCOS across
# all 450 deterministic configurations, then visualises the cost landscape
# using heatmaps: PV deployment scenario (rows) × BESS deployment scenario (columns).
# Darker cells = lower LCOE = more cost-effective configuration.
# The final filter selects Fixed tracker + IBC module (best-performing technology combo).


# For LCOE and LCOS (already numeric)
lcoe_stats = results2_df['LCOE (EUR/MWh)'].agg(['mean', 'max', 'min'])
lcos_stats = results2_df['LCOS (EUR/MWh)'].agg(['mean', 'max', 'min'])

print("Summary Statistics (Numeric Columns Only):")
print(summary_stats)

print("\nAverage Values:")
print(average_values)

print("\nMaximum Values:")
print(max_values)

print("\nLCOE Statistics:")
print(lcoe_stats)

print("\nLCOS Statistics:")
print(lcos_stats)


# Get the 4 lowest LCOE cases
lowest_lcoe_cases = results2_df.nsmallest(4, 'LCOE (EUR/MWh)')

# Print detailed results for these cases
print("TOP 4 LOWEST LCOE CONFIGURATIONS:")
print(lowest_lcoe_cases.to_string(max_colwidth=30, index=False))

# If you want to append these to a new list (e.g., for further processing)
top_4_results_append = []

for _, row in lowest_lcoe_cases.iterrows():
    top_4_results_append.append({
        # Scenario Info
        "Scenario PV": row["Scenario PV"],
        "Scenario Battery": row["Scenario Battery"],
        "Tracker Type": row["Tracker Type"],
        "Module Type": row["Module Type"],
        
        # Technical Specs
        "Battery Capacity (MW)": row["Battery Capacity (MW)"],
        "Storage Hours": row["Storage Hours"],
        "Total Solar Capacity (MW)": row["Total Solar Capacity (MW)"],
        "Total Battery Capacity (MWh)": row["Total Battery Capacity (MWh)"],
        
        # Financial Metrics
        "Solar CAPEX (EUR)": row["Solar CAPEX (EUR)"],
        "Battery CAPEX (EUR)": row["Battery CAPEX (EUR)"],
        "Total CAPEX (EUR)": row["Total CAPEX (EUR)"],
        "LCOE (EUR/MWh)": row["LCOE (EUR/MWh)"],
        "LCOS (EUR/MWh)": row["LCOS (EUR/MWh)"],
        
        # Additional calculated fields can be added here
        "CAPEX per MWh": row["Total CAPEX (EUR)"] / (row["Total Solar Capacity (MW)"] * 1000)  # Example derived metric
    })

# Convert to DataFrame
top_4_df = pd.DataFrame(top_4_results_append)
top_4_df.to_csv('top_4_lcoe_configurations.csv', index=False)


# Create clear labels for battery scenarios
battery_labels = {
    "60 MW in Year 0": "60MW at Y0",
    "50 MW in Year 0 and 10 MW in Year 1": "50MW-Y0 + 10MW-Y1",
    "50 MW first year, 5 MW next two years": "50MW-Y0 + 5MW-Y1-2",
    "20 MW first year, 20 MW next 2 years": "20MW-Y0-2",
    "10 MW first year, 10 MW next 2 years": "10MW-Y0-2",
    "30 MW first year, 30 MW next 1 year": "30MW-Y0-1",
    "10 MW first year, 20 MW next 1 year, 30": "10MW-Y0-20-30" #new
}

# Create clear labels for solar scenarios
solar_labels = {
    "100 MW in Year 0": "100MW-Y0",
    "75 MW first year, 25 MW next year": "75MW-Y0 + 25MW-Y1",
    "50 MW in Year 0 and 50 MW in Year 1": "50MW-Y0 + 50MW-Y1",
    "50 MW first year, 25 MW next two years": "50MW-Y0 + 25MW-Y1-2",
    "25 MW first year, 25 MW next years": "25MW-Y0-3"
}

# Apply the labels to your DataFrame
results2_df['Battery Scenario'] = results2_df['Scenario Battery'].map(battery_labels)
results2_df['Solar Scenario'] = results2_df['Scenario PV'].map(solar_labels)

# Aggregate to get minimum LCOE for each Solar-Battery combination
heatmap_data = results2_df.groupby(['Solar Scenario', 'Battery Scenario'])['LCOE (EUR/MWh)'].mean().unstack()

plt.figure(figsize=(14, 10))
ax = sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".1f",
    cmap="YlOrRd_r",  # Darker = better (lower LCOE)
    linewidths=0.5,
    cbar_kws={'label': 'LCOE (EUR/MWh)'}
)

# Customize the plot
plt.title("Optimal LCOE for Solar-Battery Combinations\n(Darker Colors = Lower Cost)", pad=20, fontsize=14)
plt.xlabel("Battery Installation Scenario", fontsize=12)
plt.ylabel("Solar Installation Scenario", fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=10)

# Adjust layout and save
plt.tight_layout()
#plt.savefig('solar_battery_lcoe_heatmap_step1.png', dpi=900, bbox_inches='tight')
plt.show()


###Heat MAP 2

# ======================================================================
# 1. Define Label Dictionaries (with desired display order)
# ======================================================================
battery2_labels = {
    "60 MW in Year 0": "60MW at Y0",
    "50 MW in Year 0 and 10 MW in Year 1": "50MW-Y0 + 10MW-Y1",
    "30 MW first year, 30 MW next 1 year": "30MW-Y0-1",
    "20 MW first year, 20 MW next 2 years": "20MW-Y0-2",
    "10 MW first year, 20 MW next 1 year, 30": "10MW-Y0-20-30"
}

solar2_labels = {
    "100 MW in Year 0": "100MW-Y0",
    "75 MW first year, 25 MW next year": "75MW-Y0 + 25MW-Y1",
    "50 MW in Year 0 and 50 MW in Year 1": "50MW-Y0 + 50MW-Y1",
    "50 MW first year, 25 MW next two years": "50MW-Y0 + 25MW-Y1-2",
    "25 MW first year, 25 MW next years": "25MW-Y0-3"
}

# ======================================================================
# 2. Prepare DataFrame with Correct Ordering
# ======================================================================
# Apply labels
results2_df['Battery Scenario'] = results2_df['Scenario Battery'].map(battery2_labels)
results2_df['Solar Scenario'] = results2_df['Scenario PV'].map(solar2_labels)

# Define explicit ordering (using original keys)
solar_order = [
    "100 MW in Year 0",
    "75 MW first year, 25 MW next year", 
    "50 MW in Year 0 and 50 MW in Year 1",
    "50 MW first year, 25 MW next two years",
    "25 MW first year, 25 MW next years"
]

battery_order = [
    "60 MW in Year 0",
    "50 MW in Year 0 and 10 MW in Year 1",
    "30 MW first year, 30 MW next 1 year",
    "20 MW first year, 20 MW next 2 years",
    "10 MW first year, 20 MW next 1 year, 30"
]

# Convert to labeled versions
solar_order_labels = [solar2_labels[key] for key in solar_order]
battery_order_labels = [battery2_labels[key] for key in battery_order]

# Aggregate with forced ordering
heatmap_data = (
    results2_df.groupby(['Scenario PV', 'Scenario Battery'])['LCOE (EUR/MWh)']
    .mean()
    .unstack()
    .reindex(index=solar_order, columns=battery_order)  # Force order
    .rename(index=solar_labels, columns=battery_labels)  # Apply pretty labels
)

# ======================================================================
# 3. Generate the Heatmap
# ======================================================================
plt.figure(figsize=(12, 8))
ax = sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".1f",
    cmap="YlOrRd_r",  # Reverse colormap (darker = lower LCOE)
    linewidths=0.5,
    linecolor='white',
    cbar_kws={'label': 'LCOE (EUR/MWh)', 'shrink': 0.8},
    annot_kws={'size': 10}
)

# ======================================================================
# 4. Advanced Formatting
# ======================================================================
# Title and labels
plt.title(
    "Optimal LCOE by Solar + Battery Deployment Scenario\n(Darker = Lower Cost)", 
    pad=20, 
    fontsize=14,
    fontweight='bold'
)
plt.xlabel("Battery Deployment Scenario", fontsize=12, labelpad=10)
plt.ylabel("Solar Deployment Scenario", fontsize=12, labelpad=10)

# Axis ticks
ax.set_xticklabels(battery_order_labels, rotation=45, ha='right', fontsize=10)
ax.set_yticklabels(solar_order_labels, rotation=0, fontsize=10)

# Tight layout and save
plt.tight_layout()
#plt.savefig('solar_battery_lcoe_heatmap_ordered.png', dpi=300, bbox_inches='tight')
plt.show()

####------------------------------------------
##### Just presenting the Fixed, IBC
####------------------------------------------

# Filter the DataFrame for Fixed axis and IBC module cases
filtered_summarized_df = results2_df[
    (results2_df['Tracker Type'] == 'Fixed') & 
    (results2_df['Module Type'] == 'IBC')
]

# Select only numeric columns from the filtered DataFrame
numeric_filtered_df = filtered_summarized_df.select_dtypes(include=['float64', 'int64'])

# Now compute statistics for the filtered cases
summary_stats_filtered = numeric_filtered_df.describe()
average_values_filtered = numeric_filtered_df.mean()
max_values_filtered = numeric_filtered_df.max()
min_values_filtered = numeric_filtered_df.min()

# For LCOE and LCOS in filtered cases
lcoe_stats_filtered = filtered_summarized_df['LCOE (EUR/MWh)'].agg(['mean', 'max', 'min'])
lcos_stats_filtered = filtered_summarized_df['LCOS (EUR/MWh)'].agg(['mean', 'max', 'min'])

print("\nStatistics for Fixed axis and IBC module cases only:")
print("Summary Statistics:")
print(summary_stats_filtered)

print("\nAverage Values:")
print(average_values_filtered)

print("\nMaximum Values:")
print(max_values_filtered)

print("\nMinimum Values:")
print(min_values_filtered)

print("\nLCOE Statistics for Fixed/IBC:")
print(lcoe_stats_filtered)

print("\nLCOS Statistics for Fixed/IBC:")
print(lcos_stats_filtered)

####------------------------------------------
##### HEATMAP FOR IBC FIXED
####------------------------------------------

# Filter for Fixed axis and IBC module cases
fixed_ibc_df = results2_df[
    (results2_df['Tracker Type'] == 'Fixed') & 
    (results2_df['Module Type'] == 'IBC')
]

battery2_labels_filtered = {
    "60 MW in Year 0": "60MW at Y0",
    "50 MW in Year 0 and 10 MW in Year 1": "50MW-Y0 + 10MW-Y1",
    "30 MW first year, 30 MW next 1 year": "30MW-Y0-1",
    "20 MW first year, 20 MW next 2 years": "20MW-Y0-2",
    "10 MW first year, 20 MW next 1 year, 30": "10MW-Y0-20-30"
}

solar2_labels_filtered = {
    "100 MW in Year 0": "100MW-Y0",
    "75 MW first year, 25 MW next year": "75MW-Y0 + 25MW-Y1",
    "50 MW in Year 0 and 50 MW in Year 1": "50MW-Y0 + 50MW-Y1",
    "50 MW first year, 25 MW next two years": "50MW-Y0 + 25MW-Y1-2",
    "25 MW first year, 25 MW next years": "25MW-Y0-3"
}

# Define explicit ordering (using original keys)
solar_order_filtered = [
    "100 MW in Year 0",
    "75 MW first year, 25 MW next year", 
    "50 MW in Year 0 and 50 MW in Year 1",
    "50 MW first year, 25 MW next two years",
    "25 MW first year, 25 MW next years"
]

battery_order_filtered = [
    "60 MW in Year 0",
    "50 MW in Year 0 and 10 MW in Year 1",
    "30 MW first year, 30 MW next 1 year",
    "20 MW first year, 20 MW next 2 years",
    "10 MW first year, 20 MW next 1 year, 30"
]

# Aggregate with forced ordering (using the filtered DataFrame)
heatmap_data = (
    fixed_ibc_df.groupby(['Scenario PV', 'Scenario Battery'])['LCOE (EUR/MWh)']
    .mean()
    .unstack()
    .reindex(index=solar_order_filtered, columns=battery_order_filtered)  # Force order
    .rename(index=solar2_labels_filtered, columns=battery2_labels_filtered)  # Apply pretty labels
)

# ======================================================================
# 3. Generate the Heatmap
# ======================================================================
plt.figure(figsize=(12, 8))
ax = sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".1f",
    cmap="YlOrRd_r",  # Reverse colormap (darker = lower LCOE)
    linewidths=0.5,
    linecolor='white',
    cbar_kws={'label': 'LCOE (EUR/MWh)', 'shrink': 0.8},
    annot_kws={'size': 10}
)

# ======================================================================
# 4. Advanced Formatting (with updated title)
# ======================================================================
plt.title(
    "LCOE for Fixed Axis + IBC Module Configurations\n(Darker = Lower Cost)", 
    pad=20, 
    fontsize=14,
    fontweight='bold'
)
plt.xlabel("Battery Deployment Scenario", fontsize=12, labelpad=10)
plt.ylabel("Solar Deployment Scenario", fontsize=12, labelpad=10)

# Axis ticks
ax.set_xticklabels([battery2_labels_filtered[key] for key in battery_order_filtered], rotation=45, ha='right', fontsize=10)
ax.set_yticklabels([solar2_labels_filtered[key] for key in solar_order_filtered], rotation=0, fontsize=10)

plt.tight_layout()
plt.show()


####--------------------


---
## Step 2 — Stochastic LCOE/LCOS via Monte Carlo Simulation

**Paper reference: Section 2.1.2, Eqs. (20)–(23)**

This step extends the deterministic model by introducing **uncertainty** in:

1. **Effective Availability (EA)** — the fraction of potential generation lost to curtailment,
   grid constraints, and operational disruptions. EA is sampled from a **triangular distribution**
   bounded by historical baseline and higher-curtailment projections — Eq. (25).

2. **Technology costs** — module costs, tracker costs, BESS component costs, and O&M costs
   are each perturbed by ±10–20% via triangular distributions, representing market volatility
   and supply chain uncertainty. These are the stochastic cost multipliers **Υ_PV** and **Υ_BESS** — Eq. (29).

The Monte Carlo simulation generates S = `monte_carlo_iterations_step2` scenarios.
Each scenario produces a different LCOE realisation, building the **distribution** of
expected lifecycle costs. This is the **E[LCOE]** of Eq. (20).

### Key insight from the paper (Section 1):
> "A system optimised for expected conditions... is not equivalent to a system designed
> to perform well across the full distribution of possible conditions."

The distributional outputs (CDFs, histograms) from this step reveal which configurations
are **robust** under uncertainty vs. which are sensitive to unfavourable realisations.


In [ ]:
# =============================================================================
# STEP 2: STOCHASTIC MONTE CARLO SIMULATION — E[LCOE] under uncertainty
# =============================================================================
# Extends the deterministic Step 1 model by introducing stochastic variation in:
#
#   1. EFFECTIVE AVAILABILITY (EA): Sampled via triangular distribution from
#      historical curtailment bounds — represents grid constraints, outages,
#      and operational disruptions. This is the key physical uncertainty
#      that existing ROA literature often neglects (see Section 1, para 2).
#
#   2. TECHNOLOGY COSTS: Module, tracker, and BESS component costs are each
#      perturbed by ±10-20% via triangular distributions — the stochastic
#      cost multipliers Υ_PV and Υ_BESS from Eq. (29).
#
#   3. O&M COSTS: Operational expenditures similarly perturbed.
#
# Each Monte Carlo iteration produces one LCOE/LCOS realisation.
# The collection of S realisations builds the distribution needed
# for E[LCOE] — Eq. (20) — and risk metrics (P5, P95, VaR).

##### STEP 2
###---------------------

print("Start of Step 2")

# -------------------------------------------------------------------------
# Effective Availability (EA) projection bounds.
# These values define the lower (baseline) and upper (higher) envelopes
# of the EA distribution over the 25-year horizon. They were derived
# from a separate preprocessing analysis of historical curtailment and
# grid-constraint data for the case-study region (see paper, Section 3).
# At each simulation year, EA is sampled from a triangular distribution
# bounded by these envelopes — Eq. (25).
# -------------------------------------------------------------------------

years_in_operation = np.arange(1, 26)  # Years 1 through 25
#EA_baseline = np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0,0.0, 0.0, 0.0, 0.0, 0.0,0.0, 0.0, 0.0, 0.0, 0.0,0.0, 0.0, 0.0, 0.0, 0.0])
#EA_higher = np.array([0.2, 0.2, 0.2, 0.2, 0.2,0.2, 0.2, 0.2, 0.2, 0.2,0.2, 0.2, 0.2, 0.2, 0.2,0.2, 0.2, 0.2, 0.2, 0.2,0.2, 0.2, 0.2, 0.2, 0.2])
EA_baseline = np.array([0.0118,0.0043,0.0033, 0.0089, 0.0036, 0.0034, 0.0031, 0.0030, 0.0032, 0.0027, 0.0025, 0.0024, 0.0024, 0.0022, 0.0021, 0.0023, 0.0018, 0.0016, 0.0015, 0.0011, 0.0010, 0.0007, 0.0007, 0.0007, 0.0007])
EA_higher = np.array([0.1191, 0.1413, 0.1604, 0.1430, 0.1489, 0.1437, 0.1633, 0.1558, 0.1704, 0.1731, 0.2125, 0.2427, 0.2277, 0.2038, 0.2058, 0.2200, 0.2274, 0.2301, 0.2402, 0.2403, 0.2339,0.2402,0.2402,0.2402,0.2402])
EA_mean = (EA_baseline + EA_higher) / 2

# Create interpolation functions
f_baseline = interp1d(years_in_operation, EA_baseline, 
                     kind='linear', fill_value="extrapolate")
f_mean = interp1d(years_in_operation, EA_mean, kind='linear', fill_value="extrapolate")
f_higher = interp1d(years_in_operation, EA_higher,
                   kind='linear', fill_value="extrapolate")

# ========== CURTAILMENT FUNCTIONS ==========
def get_EA(year_of_operation):
    baseline = f_baseline(year_of_operation)
    higher = f_higher(year_of_operation)
    mean = f_mean(year_of_operation)  # Evaluate f_mean at year_of_operation
    random_EA = np.random.triangular(baseline, mean, higher)   #Best results obtained with EA ranging between 0 and 20%
    return random_EA


# Installation Scenarios
installation_scenarios_step2 = {
    "100 MW in Year 0": [100] + [0] * (project_lifetime),
    "50 MW in Year 0 and 50 MW in Year 1": [50, 50] + [0] * (project_lifetime - 1),
    "50 MW first year, 25 MW next two years": [50, 25, 25] + [0] * (project_lifetime - 2),
    "75 MW first year, 25 MW next year": [75, 25] + [0] * (project_lifetime - 1),
    "25 MW first year, 25 MW next years": [25, 25, 25, 25] + [0] * (project_lifetime - 3),
}



# Battery storage durations (in hours)
storage_durations_step2 = [1] #,2,3,4 , 5, 6, 7, 8, 9, 10]  # Example storage durations #Can be modified according to the news of the user

battery_installation_scenarios_step2 = {
    "60 MW in Year 0": [60] + [0] * (project_lifetime),
    "50 MW in Year 0 and 10 MW in Year 1": [50, 10] + [0] * (project_lifetime - 1),
    "20 MW first year, 20 MW next 2 years": [20,20,20] + [0] * (project_lifetime - 2),
    "30 MW first year, 30 MW next 1 year": [30, 30] + [0] * (project_lifetime - 1),
    "10 MW first year, 20 MW next 1 year, 30": [10, 20, 30] + [0] * (project_lifetime - 2), 
    #relation between scenarios battery and solar PV
}

monte_carlo_iterations_step2 = 2000 #can be modified according to the needs of the user

results_step2 = []
for simulation in range(monte_carlo_iterations_step2):
    for scenario_name_battery, installation_schedule_battery in battery_installation_scenarios_step2.items():    
        total_battery_capacity_mw = sum(installation_schedule_battery)
        for storage_hours in storage_durations_step2:      
            #print(storage_hours)
            storage_hours_rounded = int(round(storage_hours))
            for scenario_name, installation_schedule in installation_scenarios_step2.items():
                for tracker, tracker_cost_dict in tracking_costs.items():
                    tracker_multiplier = tracker_multipliers[tracker]
                    
                    tracker_opex_per_kw_step2 = np.random.triangular(
                        0.8 * opex_per_kw_per_year[tracker],
                        opex_per_kw_per_year[tracker],
                        1.2 * opex_per_kw_per_year[tracker]
                    )
                    
                    for module, module_info in module_data.items():
                        last_installation_year = max(idx for idx, cap in enumerate(installation_schedule) if cap > 0)
                        extended_project_lifetime = last_installation_year + 21
                        module_efficiency_multiplier = module_efficiency[module]
        
                        # INITIALIZE ALL FINANCIAL TRACKERS
                        total_solar_capex = 0.0
                        total_battery_capex = 0.0
                        npv_numerator_lcoe = 0.0
                        npv_denominator_lcoe = 0.0
                        npv_numerator_lcos = 0.0
                        npv_denominator_lcos = 0.0
                        cumulative_installed_capacity_mw = 0
                        cumulative_battery_installed_capacity_mw = 0
                        active_generations = [0] * extended_project_lifetime
                        battery_generations_discharge = [0] * extended_project_lifetime
                        Salvage_total_lcoe=0
                        Salvage_total_lcos=0
                        # Initialize decommissioned capacities
                        decommissioned_capacity = 0
                        decommissioned_capacity_battery = 0
                        
        
                        for year in range(extended_project_lifetime):
                            discount_factor = 1 / ((1 + discount_rate) ** year) if year > 0 else 1
                            
                            # CAPACITY TRACKING
                            installed_capacity_mw = installation_schedule[year] if year < len(installation_schedule) else 0
                            battery_installed_capacity_mw = installation_schedule_battery[year] if year < len(installation_schedule_battery) else 0
                            
                            if year >= 20:
                                decommissioned_capacity = installation_schedule[year - 20]
                                cumulative_installed_capacity_mw -= decommissioned_capacity
                                decommissioned_capacity_battery = installation_schedule_battery[year - 20]
                                cumulative_battery_installed_capacity_mw -= decommissioned_capacity_battery
                            
                            cumulative_installed_capacity_mw += installed_capacity_mw
                            cumulative_battery_installed_capacity_mw += battery_installed_capacity_mw
                            
                            # SOLAR CAPEX
                            if installed_capacity_mw > 0:
                                cost_category = determine_cost_category(installed_capacity_mw)
                                tracker_cost_per_watt_step2 = np.random.triangular(
                                0.8*tracker_cost_dict[cost_category],
                                tracker_cost_dict[cost_category],
                                1.2*tracker_cost_dict[cost_category]
                                )

                                module_cost_per_watt_step2 = np.random.triangular(
                                0.8*module_info["cost_per_watt"],
                                module_info["cost_per_watt"],
                                1.2*module_info["cost_per_watt"]
                                )
                                
                                lf_tech = (1 - Learning_rate_technology) ** max(0, year - 1)
                                lf_exp = (1 - Learning_rate_experience) ** max(0, year - 1)
                                
                                
                                plant_capacity_w = installed_capacity_mw * 1e6
                                module_cost = plant_capacity_w * module_cost_per_watt_step2 * lf_tech
                                tracker_cost = plant_capacity_w * tracker_cost_per_watt_step2 * lf_exp
                                yearly_solar_capex = module_cost + tracker_cost
                                total_solar_capex += yearly_solar_capex * discount_factor
                            
                            # Determine cost dict based on INSTALLED capacity (not total)
                            if battery_installed_capacity_mw < 10:
                                battery_cost_dict = cost_dict
                            elif battery_installed_capacity_mw < 61:
                                battery_cost_dict = cost_dict_10
                            else:
                                battery_cost_dict = cost_dict_100
                            
                            lf_BESS = (1 - learning_rate_BESS1) ** max(0, year - 1)
                            
                            # Use INSTALLED capacity (not total)
                            battery_capacity_kwh = battery_installed_capacity_mw * 1e3 * storage_hours_rounded
                            battery_capacity_kw = battery_installed_capacity_mw * 1e3
                            
                            battery_cost = battery_capacity_kwh * (
                                np.random.triangular(
                                    0.9 * battery_cost_dict[f"{storage_hours_rounded} hours"]["DC Storage Block ($/kWh)"],
                                    battery_cost_dict[f"{storage_hours_rounded} hours"]["DC Storage Block ($/kWh)"],
                                    1.1 * battery_cost_dict[f"{storage_hours_rounded} hours"]["DC Storage Block ($/kWh)"]
                                    ) +
                                    np.random.triangular(
                                    0.9 * battery_cost_dict[f"{storage_hours_rounded} hours"]["DC Storage BOS ($/kWh)"],
                                    battery_cost_dict[f"{storage_hours_rounded} hours"]["DC Storage BOS ($/kWh)"],
                                    1.1 * battery_cost_dict[f"{storage_hours_rounded} hours"]["DC Storage BOS ($/kWh)"]
                                    ) +
                                    np.random.triangular(
                                    0.9 * battery_cost_dict[f"{storage_hours_rounded} hours"]["Systems Integration ($/kWh)"],
                                    battery_cost_dict[f"{storage_hours_rounded} hours"]["Systems Integration ($/kWh)"],
                                    1.1 * battery_cost_dict[f"{storage_hours_rounded} hours"]["Systems Integration ($/kWh)"]
                                    ) +
                                    np.random.triangular(
                                    0.9 * battery_cost_dict[f"{storage_hours_rounded} hours"]["EPC ($/kWh)"],
                                    battery_cost_dict[f"{storage_hours_rounded} hours"]["EPC ($/kWh)"],
                                    1.1 * battery_cost_dict[f"{storage_hours_rounded} hours"]["EPC ($/kWh)"]
                                    ) +
                                    np.random.triangular(
                                    0.9 * battery_cost_dict[f"{storage_hours_rounded} hours"]["Project Development ($/kWh)"],
                                    battery_cost_dict[f"{storage_hours_rounded} hours"]["Project Development ($/kWh)"],
                                    1.1 * battery_cost_dict[f"{storage_hours_rounded} hours"]["Project Development ($/kWh)"]
                                    ))* Dolar * lf_BESS + battery_capacity_kw * (
                                    np.random.triangular(
                                    0.9 * battery_cost_dict[f"{storage_hours_rounded} hours"]["Power Equipment ($/kW)"],
                                    battery_cost_dict[f"{storage_hours_rounded} hours"]["Power Equipment ($/kW)"],
                                    1.1 * battery_cost_dict[f"{storage_hours_rounded} hours"]["Power Equipment ($/kW)"]
                                    )+
                                    np.random.triangular(
                                    0.9 * battery_cost_dict[f"{storage_hours_rounded} hours"]["CC ($/kW)"],
                                    battery_cost_dict[f"{storage_hours_rounded} hours"]["CC ($/kW)"],
                                    1.1 * battery_cost_dict[f"{storage_hours_rounded} hours"]["CC ($/kW)"]
                                    )+
                                    np.random.triangular(
                                    0.9 * battery_cost_dict[f"{storage_hours_rounded} hours"]["Grid Integration ($/kW)"],
                                    battery_cost_dict[f"{storage_hours_rounded} hours"]["Grid Integration ($/kW)"],
                                    1.1 * battery_cost_dict[f"{storage_hours_rounded} hours"]["Grid Integration ($/kW)"]
                                    )) * Dolar*lf_BESS
                    
                            total_battery_capex += battery_cost * discount_factor
                            
                             #Active Generation (for LCOE)
                            raw_generations_for_year=0
                            if year>0:
                                #active_generation_for_year = 0 #original modified
                                for previous_year in range(max(0, year - 19), year + 1):
                                    active_capacity_mw = installation_schedule[previous_year] if previous_year < len(installation_schedule) else 0
                                    degradation_factor = (1 - degradation_rate) ** (year - previous_year)
                                    raw_generations_for_year += (     #active_generation_for_year
                                        P_profiles[min(year, len(P_profiles) - 1)]
                                        * active_capacity_mw / Power_capacity
                                        * tracker_multiplier
                                        * module_efficiency_multiplier
                                        * degradation_factor
                                    )
                                    EA_pct = get_EA(min(year + 1, 25)) #these lines are new
                                    active_generation_for_year = raw_generations_for_year * (1 - EA_pct)
                                
    
                                # Battery Discharge (for LCOS)
                                cycles_per_year = 4500 / 20  # 4500 cycles over 20 years

                                battery_generation_for_year = 0
            
                                for previous_year in range(max(0, year - 19), year + 1):
                                    battery_active_capacity_mw = installation_schedule_battery[previous_year] if previous_year < len(installation_schedule_battery) else 0
                                    degradation_battery_factor = (1 - Battery_degradation1) ** (year - previous_year)
                                    battery_generation_for_year += (
                                        battery_active_capacity_mw
                                        * storage_hours_rounded
                                        * battery_efficiency
                                        * DoD
                                        * degradation_battery_factor
                                        * cycles_per_year
                                    )
                                    
                                npv_numerator_lcoe += (
                                tracker_opex_per_kw_step2 * cumulative_installed_capacity_mw * 1e3 * discount_factor +
                                Land_Renting / 20 * installed_capacity_mw * discount_factor +
                                np.random.triangular(
                                    0.9 * battery_cost_dict[f"{storage_hours_rounded} hours"]["Operation Cost ($/kW-year)"],
                                    battery_cost_dict[f"{storage_hours_rounded} hours"]["Operation Cost ($/kW-year)"],
                                    1.1 * battery_cost_dict[f"{storage_hours_rounded} hours"]["Operation Cost ($/kW-year)"]
                                    ) * battery_capacity_kw * discount_factor)
                            
                                # LCOS Numerator (costs)
                                npv_numerator_lcos += np.random.triangular(
                                    0.9 * battery_cost_dict[f"{storage_hours_rounded} hours"]["Operation Cost ($/kW-year)"],
                                    battery_cost_dict[f"{storage_hours_rounded} hours"]["Operation Cost ($/kW-year)"],
                                    1.1 * battery_cost_dict[f"{storage_hours_rounded} hours"]["Operation Cost ($/kW-year)"]
                                    ) * battery_capacity_kw * discount_factor

                                active_generations[year] = active_generation_for_year 
                                battery_generations_discharge[year] = battery_generation_for_year

                            
                                # LCOE Denominator (energy generation)
                                npv_denominator_lcoe += (active_generations[year] + battery_generations_discharge[year])* discount_factor  #active generations [year] before
           
                                # LCOS Denominator (energy discharged)
                                npv_denominator_lcos += battery_generations_discharge[year] * discount_factor
        
                            
                            if decommissioned_capacity > 0:
                                salvage_year = Salvage * decommissioned_capacity * discount_factor
                                Salvage_total_lcoe += salvage_year
                            if decommissioned_capacity_battery > 0:
                                salvage_year_battery = np.random.triangular(
                                    0.9 * Salvage_battery,
                                    Salvage_battery,
                                    1.1 * Salvage_battery) * 1e3 * storage_hours_rounded * decommissioned_capacity_battery * discount_factor
                                Salvage_total_lcos += salvage_year_battery

                        salvage_total = Salvage_total_lcoe + Salvage_total_lcos
                        salvage_total_battery = Salvage_total_lcos


                        
                        # FINAL METRICS CALCULATION
                        lcoe = (total_solar_capex + total_battery_capex + npv_numerator_lcoe+salvage_total) / npv_denominator_lcoe if npv_denominator_lcoe != 0 else float("inf")
                        lcos = (total_battery_capex + npv_numerator_lcos+salvage_total_battery) / npv_denominator_lcos if npv_denominator_lcos != 0 else float("inf")
                        
                        # RESULTS APPEND
                        results_step2.append({
                            "Scenario PV": scenario_name,
                            "Scenario Battery": scenario_name_battery,
                            "Tracker Type": tracker,
                            "Module Type": module,
                            "Battery Capacity (MW)": total_battery_capacity_mw,
                            "Storage Hours": storage_hours_rounded,
                            
                            # CAPEX COMPONENTS
                            "Solar CAPEX (EUR)": total_solar_capex,
                            "Battery CAPEX (EUR)": total_battery_capex,
                            "Total CAPEX (EUR)": total_solar_capex + total_battery_capex,
                            
                            # LCOE METRICS
                            "LCOE (EUR/MWh)": lcoe,
                            "LCOS (EUR/MWh)": lcos,
                            
                            # TECHNICAL METRICS
                            "Total Solar Capacity (MW)": sum(installation_schedule),
                            "Total Battery Capacity (MWh)": total_battery_capacity_mw * storage_hours_rounded
                        })
    
    # Convert to DataFrame
results_step2_df = pd.DataFrame(results_step2)

# Select only numeric columns
numeric2_df = results_step2_df.select_dtypes(include=['float64', 'int64'])

# Now compute statistics safely
summary_stats = numeric2_df.describe()
average_values = numeric2_df.mean()
max_values = numeric2_df.max()
min_values = numeric2_df.min()

# For LCOE and LCOS (already numeric)
lcoe_stats = results_step2_df['LCOE (EUR/MWh)'].agg(['mean', 'max', 'min'])
lcos_stats = results_step2_df['LCOS (EUR/MWh)'].agg(['mean', 'max', 'min'])

print("Summary Statistics (Numeric Columns Only):")
print(summary_stats)

print("\nAverage Values:")
print(average_values)

print("\nMaximum Values:")
print(max_values)

print("\nLCOE Statistics:")
print(lcoe_stats)

print("\nLCOS Statistics:")
print(lcos_stats)

# Create clear labels for battery scenarios
battery_labels_step2 = {
    "60 MW in Year 0": "60MW at Y0",
    "50 MW in Year 0 and 10 MW in Year 1": "50MW-Y0 + 10MW-Y1",
    "50 MW first year, 5 MW next two years": "50MW-Y0 + 5MW-Y1-2",
    "20 MW first year, 20 MW next 2 years": "20MW-Y0-2",
    "10 MW first year, 10 MW next 2 years": "10MW-Y0-2",
    "30 MW first year, 30 MW next 1 year": "30MW-Y0-1",
    "10 MW first year, 20 MW next 1 year, 30": "10MW-Y0-20-30" #new
}

# Create clear labels for solar scenarios
solar_labels_step2 = {
    "100 MW in Year 0": "100MW-Y0",
    "75 MW first year, 25 MW next year": "75MW-Y0 + 25MW-Y1",
    "50 MW in Year 0 and 50 MW in Year 1": "50MW-Y0 + 50MW-Y1",
    "50 MW first year, 25 MW next two years": "50MW-Y0 + 25MW-Y1-2",
    "25 MW first year, 25 MW next years": "25MW-Y0-3"
}

# Apply the labels to your DataFrame
results_step2_df['Battery Scenario'] = results_step2_df['Scenario Battery'].map(battery_labels_step2)
results_step2_df['Solar Scenario'] = results_step2_df['Scenario PV'].map(solar_labels_step2)

# Aggregate to get minimum LCOE for each Solar-Battery combination
heatmap_step2_data = results_step2_df.groupby(['Solar Scenario', 'Battery Scenario'])['LCOE (EUR/MWh)'].mean().unstack()

plt.figure(figsize=(14, 10))
ax = sns.heatmap(
    heatmap_step2_data,
    annot=True,
    fmt=".1f",
    cmap="YlOrRd_r",  # Darker = better (lower LCOE)
    linewidths=0.5,
    cbar_kws={'label': 'LCOE (EUR/MWh)'}
)

# Customize the plot
plt.title("Optimal LCOE for Solar-Battery Combinations\n(Darker Colors = Lower Cost)", pad=20, fontsize=14)
plt.xlabel("Battery Installation Scenario", fontsize=12)
plt.ylabel("Solar Installation Scenario", fontsize=12)
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.yticks(rotation=0, fontsize=10)

# Adjust layout and save
plt.tight_layout()
#plt.savefig('solar_battery_lcoe_heatmap_step1.png', dpi=900, bbox_inches='tight')
plt.show()


### Step 2 — Stochastic Results: Heatmaps, CDFs, and Histograms

The visualisations below show:
1. **Heatmaps** of mean E[LCOE] across deployment scenarios under uncertainty
2. **CDFs** showing the full probability distribution of LCOE for each configuration
3. **Histograms** with P5/P95 risk markers for downside/upside exposure

The CDF comparison is central to the Strategic Engineering methodology:
configurations whose CDFs are shifted left (lower LCOE) across the full probability
range are **stochastically dominant** — they perform better under virtually all scenarios.


In [ ]:
# =============================================================================
# STEP 2 RESULTS — Heatmaps, CDFs, Histograms, and Filtered Analysis
# =============================================================================
# This section produces:
#   1. Heatmaps of mean E[LCOE] for all PV × BESS scenario combinations
#   2. CDF plots comparing the full LCOE distribution across configurations
#      → Configurations with CDFs shifted LEFT are stochastically dominant
#   3. Histograms with P5/P95 risk markers for each configuration
#   4. Filtered analysis for Fixed tracker + IBC module (best technology combo)
#   5. CSV export of top-25 configurations for external analysis
#
# The CDF comparison is central to the Strategic Engineering methodology:
# it reveals which deployment strategies are ROBUST under uncertainty
# vs. which are sensitive to adverse realisations (right-tail risk).

###End of Step 2

###Heat MAP 2

# ======================================================================
# 1. Define Label Dictionaries (with desired display order)
# ======================================================================
battery2_labels_step2 = {
    "60 MW in Year 0": "60MW at Y0",
    "50 MW in Year 0 and 10 MW in Year 1": "50MW-Y0 + 10MW-Y1",
    "30 MW first year, 30 MW next 1 year": "30MW-Y0-1",
    "20 MW first year, 20 MW next 2 years": "20MW-Y0-2",
    "10 MW first year, 20 MW next 1 year, 30": "10MW-Y0-20-30"
}

solar2_labels_step2 = {
    "100 MW in Year 0": "100MW-Y0",
    "75 MW first year, 25 MW next year": "75MW-Y0 + 25MW-Y1",
    "50 MW in Year 0 and 50 MW in Year 1": "50MW-Y0 + 50MW-Y1",
    "50 MW first year, 25 MW next two years": "50MW-Y0 + 25MW-Y1-2",
    "25 MW first year, 25 MW next years": "25MW-Y0-3"
}

# ======================================================================
# 2. Prepare DataFrame with Correct Ordering
# ======================================================================
# Apply labels
results_step2_df['Battery Scenario'] = results_step2_df['Scenario Battery'].map(battery2_labels_step2)
results_step2_df['Solar Scenario'] = results_step2_df['Scenario PV'].map(solar2_labels_step2)

# Define explicit ordering (using original keys)
solar_order_step2 = [
    "100 MW in Year 0",
    "75 MW first year, 25 MW next year", 
    "50 MW in Year 0 and 50 MW in Year 1",
    "50 MW first year, 25 MW next two years",
    "25 MW first year, 25 MW next years"
]

battery_order_step2 = [
    "60 MW in Year 0",
    "50 MW in Year 0 and 10 MW in Year 1",
    "30 MW first year, 30 MW next 1 year",
    "20 MW first year, 20 MW next 2 years",
    "10 MW first year, 20 MW next 1 year, 30"
]

# Convert to labeled versions
solar_order_labels_step2 = [solar2_labels_step2[key] for key in solar_order_step2]
battery_order_labels_step2 = [battery2_labels_step2[key] for key in battery_order_step2]

# Aggregate with forced ordering
heatmap_step2_data = (
    results_step2_df.groupby(['Scenario PV', 'Scenario Battery'])['LCOE (EUR/MWh)']
    .mean()
    .unstack()
    .reindex(index=solar_order_step2, columns=battery_order_step2)  # Force order
    .rename(index=solar_labels_step2, columns=battery_labels_step2)  # Apply pretty labels
)

# ======================================================================
# 3. Generate the Heatmap
# ======================================================================
plt.figure(figsize=(12, 8))
ax = sns.heatmap(
    heatmap_step2_data,
    annot=True,
    fmt=".1f",
    cmap="YlOrRd_r",  # Reverse colormap (darker = lower LCOE)
    linewidths=0.5,
    linecolor='white',
    cbar_kws={'label': 'LCOE (EUR/MWh)', 'shrink': 0.8},
    annot_kws={'size': 10}
)

# ======================================================================
# 4. Advanced Formatting
# ======================================================================
# Title and labels
plt.title(
    "Optimal LCOE by Solar + Battery Deployment Scenario\n(Darker = Lower Cost)", 
    pad=20, 
    fontsize=14,
    fontweight='bold'
)
plt.xlabel("Battery Deployment Scenario", fontsize=12, labelpad=10)
plt.ylabel("Solar Deployment Scenario", fontsize=12, labelpad=10)

# Axis ticks
ax.set_xticklabels(battery_order_labels_step2, rotation=45, ha='right', fontsize=10)
ax.set_yticklabels(solar_order_labels_step2, rotation=0, fontsize=10)

# Tight layout and save
plt.tight_layout()
#plt.savefig('solar_battery_lcoe_heatmap_ordered.png', dpi=300, bbox_inches='tight')
plt.show()

##### Start of the LCOE CDF ###########

# 1. Create a combined label for each configuration
results_step2_df['Configuration'] = (
    results_step2_df['Solar Scenario'] + " + " + results_step2_df['Battery Scenario']
)

# 2. Get unique configurations and assign colors/linestyles
configurations = results_step2_df['Configuration'].unique()
n_configs = len(configurations)

# Use a colormap with 25 distinct colors (e.g., 'tab20' + 'tab20b')
colors = plt.cm.tab20(np.linspace(0, 1, min(20, n_configs)))  # First 20 colors
if n_configs > 20:
    colors = np.vstack([colors, plt.cm.tab20b(np.linspace(0, 1, n_configs - 20))])

# 3. Plot CDFs
plt.figure(figsize=(16, 10))

for i, config in enumerate(configurations):
    lcoe_values = results_step2_df[results_step2_df['Configuration'] == config]['LCOE (EUR/MWh)']
    sorted_values = np.sort(lcoe_values)
    cdf = np.arange(1, len(sorted_values) + 1) / len(sorted_values)
    plt.plot(sorted_values, cdf, color=colors[i], label=config, linewidth=2)

# 4. Customize plot
plt.title('CDF of LCOE for All 25 Solar + Battery Configurations', fontsize=16, pad=20)
plt.xlabel('LCOE (EUR/MWh)', fontsize=14)
plt.ylabel('Cumulative Probability', fontsize=14)
plt.grid(True, linestyle='--', alpha=0.5)

# 5. Add a legend with 2 columns to save space
plt.legend(
    bbox_to_anchor=(1.05, 1), 
    loc='upper left', 
    borderaxespad=0., 
    fontsize=9, 
    ncol=2
)

plt.tight_layout()
plt.show()

# Group by configuration and calculate statistics
lcoe_stats = (
    results_step2_df.groupby('Configuration')['LCOE (EUR/MWh)']
    .agg(['mean', lambda x: x.quantile(0.95), lambda x: x.quantile(0.05)])
    .rename(columns={'mean': 'Mean LCOE', '<lambda_0>': 'P95', '<lambda_1>': 'P5'})
    .sort_values(by='Mean LCOE')  # Sort by mean LCOE for clarity
)

# Display the results
print(lcoe_stats)

#### HISTOGRAM

# 1. Set up the figure grid (5x5 for 25 configurations)
fig, axes = plt.subplots(5, 5, figsize=(25, 20))
fig.suptitle('LCOE Distribution per Solar + Battery Configuration', fontsize=16, y=1.02)

# 2. Flatten axes for easy iteration
axes = axes.flatten()

# 3. Plot histograms for each configuration
for i, config in enumerate(configurations):
    # Extract LCOE values for this configuration
    lcoe_values = results_step2_df[results_step2_df['Configuration'] == config]['LCOE (EUR/MWh)']
    
    # Plot histogram
    sns.histplot(
        lcoe_values,
        bins=20,
        kde=True,  # Add Kernel Density Estimate
        ax=axes[i],
        color='skyblue',
        edgecolor='black'
    )
    
    # Add vertical lines for mean/P5/P95
    mean_lcoe = lcoe_values.mean()
    p5 = lcoe_values.quantile(0.05)
    p95 = lcoe_values.quantile(0.95)
    
    axes[i].axvline(mean_lcoe, color='red', linestyle='--', label=f'Mean: {mean_lcoe:.1f}')
    axes[i].axvline(p5, color='green', linestyle=':', label=f'P5: {p5:.1f}')
    axes[i].axvline(p95, color='orange', linestyle=':', label=f'P95: {p95:.1f}')
    
    # Customize subplot
    axes[i].set_title(config, fontsize=10, pad=5)
    axes[i].set_xlabel('LCOE (EUR/MWh)', fontsize=8)
    axes[i].set_ylabel('Frequency', fontsize=8)
    axes[i].legend(fontsize=6)
    axes[i].grid(alpha=0.3)

# 4. Adjust layout
plt.tight_layout()
plt.show()

###############------------------------
###Monte Carlo Filtered
###############------------------------

Step2_fixed_ibc_df = results_step2_df[
    (results_step2_df['Tracker Type'] == 'Fixed') & 
    (results_step2_df['Module Type'] == 'IBC')
]

### CODE TO SAVE CSV

# 1. Compute mean ELCOE per configuration
config_means = (
    Step2_fixed_ibc_df.groupby('Configuration')['LCOE (EUR/MWh)'].mean().sort_values()
)

# 2. Get the 10 configurations with the lowest mean ELCOE
top25_configs = config_means.head(25).index.tolist()

# 3. Filter the DataFrame for just these configurations
cdf_export_df = Step2_fixed_ibc_df[Step2_fixed_ibc_df['Configuration'].isin(top25_configs)].copy()

# 4. Sort for neatness (optional)
cdf_export_df = cdf_export_df.sort_values(['Configuration', 'LCOE (EUR/MWh)'])

# 5. Save to CSV for Origin
cdf_export_df[['Configuration', 'LCOE (EUR/MWh)', 'LCOS (EUR/MWh)']].to_csv(
    "elcoe_elcos_top25_configurations.csv", index=True
)

print("Exported CDF data for 25 lowest ELCOE configurations to elcoe_elcos_top25_configurations.csv")
###

###Heat MAP 2

# ======================================================================
# 1. Define Label Dictionaries (with desired display order)
# ======================================================================
battery2_labels_step2_filtered = {
    "60 MW in Year 0": "60MW at Y0",
    "50 MW in Year 0 and 10 MW in Year 1": "50MW-Y0 + 10MW-Y1",
    "30 MW first year, 30 MW next 1 year": "30MW-Y0-1",
    "20 MW first year, 20 MW next 2 years": "20MW-Y0-2",
    "10 MW first year, 20 MW next 1 year, 30": "10MW-Y0-20-30"
}

solar2_labels_step2_filtered = {
    "100 MW in Year 0": "100MW-Y0",
    "75 MW first year, 25 MW next year": "75MW-Y0 + 25MW-Y1",
    "50 MW in Year 0 and 50 MW in Year 1": "50MW-Y0 + 50MW-Y1",
    "50 MW first year, 25 MW next two years": "50MW-Y0 + 25MW-Y1-2",
    "25 MW first year, 25 MW next years": "25MW-Y0-3"
}

solar_order_step2_filtered = [
    "100 MW in Year 0",
    "75 MW first year, 25 MW next year", 
    "50 MW in Year 0 and 50 MW in Year 1",
    "50 MW first year, 25 MW next two years",
    "25 MW first year, 25 MW next years"
]

battery_order_step2_filtered = [
    "60 MW in Year 0",
    "50 MW in Year 0 and 10 MW in Year 1",
    "30 MW first year, 30 MW next 1 year",
    "20 MW first year, 20 MW next 2 years",
    "10 MW first year, 20 MW next 1 year, 30"
]

# Aggregate with forced ordering (using the filtered DataFrame)
heatmap_data_step2_filtered = (
    Step2_fixed_ibc_df.groupby(['Scenario PV', 'Scenario Battery'])['LCOE (EUR/MWh)']
    .mean()
    .unstack()
    .reindex(index=solar_order_step2_filtered, columns=battery_order_step2_filtered)  # Force order
    .rename(index=solar2_labels_step2_filtered, columns=battery2_labels_step2_filtered)  # Apply pretty labels
)

# ======================================================================
# 3. Generate the Heatmap
# ======================================================================
plt.figure(figsize=(12, 8))
ax = sns.heatmap(
    heatmap_data_step2_filtered,
    annot=True,
    fmt=".1f",
    cmap="YlOrRd_r",  # Reverse colormap (darker = lower LCOE)
    linewidths=0.5,
    linecolor='white',
    cbar_kws={'label': 'LCOE (EUR/MWh)', 'shrink': 0.8},
    annot_kws={'size': 10}
)

# ======================================================================
# 4. Advanced Formatting (with updated title)
# ======================================================================
plt.title(
    "LCOE for Fixed Axis + IBC Module Configurations\n(Darker = Lower Cost)", 
    pad=20, 
    fontsize=14,
    fontweight='bold'
)
plt.xlabel("Battery Deployment Scenario", fontsize=12, labelpad=10)
plt.ylabel("Solar Deployment Scenario", fontsize=12, labelpad=10)

# Axis ticks
ax.set_xticklabels([battery2_labels_step2_filtered[key] for key in battery_order_step2_filtered], rotation=45, ha='right', fontsize=10)
ax.set_yticklabels([solar2_labels_step2_filtered[key] for key in solar_order_step2_filtered], rotation=0, fontsize=10)

plt.tight_layout()
plt.show()

###################
# NEW CDF just for IBC and Fixed cases
###################

# 1. Create a combined label for each configuration
Step2_fixed_ibc_df['Configuration'] = (
    Step2_fixed_ibc_df['Solar Scenario'] + " + " + Step2_fixed_ibc_df['Battery Scenario']
)

# 2. Get unique configurations and assign colors/linestyles
configurations = Step2_fixed_ibc_df['Configuration'].unique()
n_configs = len(configurations)

# Use a colormap with enough distinct colors
if n_configs <= 10:
    colors = plt.cm.tab10(np.linspace(0, 1, n_configs))
elif n_configs <= 20:
    colors = plt.cm.tab20(np.linspace(0, 1, n_configs))
else:
    colors = plt.cm.gist_rainbow(np.linspace(0, 1, n_configs))

# 3. Plot CDFs
plt.figure(figsize=(14, 8))

for i, config in enumerate(configurations):
    lcoe_values = Step2_fixed_ibc_df[Step2_fixed_ibc_df['Configuration'] == config]['LCOE (EUR/MWh)']
    sorted_values = np.sort(lcoe_values)
    cdf = np.arange(1, len(sorted_values) + 1) / len(sorted_values)
    plt.plot(sorted_values, cdf, color=colors[i], label=config, linewidth=2.5)

# 4. Customize plot
plt.title('CDF of LCOE for Fixed Axis + IBC Module Configurations', fontsize=16, pad=20)
plt.xlabel('LCOE (EUR/MWh)', fontsize=14)
plt.ylabel('Cumulative Probability', fontsize=14)
plt.grid(True, linestyle='--', alpha=0.5)

# 5. Add legend with 1 or 2 columns depending on number of configurations
legend_cols = 1 if n_configs <= 10 else 2
plt.legend(
    bbox_to_anchor=(1.05, 1), 
    loc='upper left', 
    borderaxespad=0., 
    fontsize=10, 
    ncol=legend_cols
)

plt.tight_layout()
plt.show()

###################
# NEW Histogram just for IBC and Fixed cases
###################

# Get unique configurations
configurations = Step2_fixed_ibc_df['Configuration'].unique()
n_configs = len(configurations)

# Calculate grid dimensions (aim for roughly square layout)
n_cols = int(np.ceil(np.sqrt(n_configs)))
n_rows = int(np.ceil(n_configs / n_cols))

# Set up the figure grid
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*4, n_rows*3))
fig.suptitle('LCOE Distribution for Fixed Axis + IBC Module Configurations', fontsize=16, y=1.02)

# Flatten axes for easy iteration
axes = axes.flatten()

# Plot histograms for each configuration
for i, config in enumerate(configurations):
    # Extract LCOE values
    lcoe_values = Step2_fixed_ibc_df[Step2_fixed_ibc_df['Configuration'] == config]['LCOE (EUR/MWh)']
    
    # Plot histogram with KDE
    sns.histplot(
        lcoe_values,
        bins=15,  # Fewer bins for cleaner look with filtered data
        kde=True,
        ax=axes[i],
        color='#1f77b4',  # Consistent blue color
        edgecolor='white',
        linewidth=0.5
    )
    
    # Add statistical markers
    mean_lcoe = lcoe_values.mean()
    p5 = lcoe_values.quantile(0.05)
    p95 = lcoe_values.quantile(0.95)
    
    axes[i].axvline(mean_lcoe, color='red', linestyle='--', linewidth=1.5, label=f'Mean: {mean_lcoe:.1f}')
    axes[i].axvline(p5, color='green', linestyle=':', linewidth=1.5, label=f'P5: {p5:.1f}')
    axes[i].axvline(p95, color='purple', linestyle=':', linewidth=1.5, label=f'P95: {p95:.1f}')
    
    # Subplot customization
    axes[i].set_title(config, fontsize=10, pad=6)
    axes[i].set_xlabel('LCOE (EUR/MWh)', fontsize=8)
    axes[i].set_ylabel('Count', fontsize=8)
    axes[i].legend(fontsize=7, framealpha=0.9)
    axes[i].grid(alpha=0.2)

# Hide empty subplots if any
for j in range(i+1, len(axes)):
    axes[j].axis('off')

# Adjust layout
plt.tight_layout()
plt.show()

###########################


---
## Step 3 — Flexible Capacity Expansion with Decision Rules (Real Options)

**Paper reference: Section 2.1.3, Eqs. (24)–(32)**

This is the core contribution: instead of committing to a fixed deployment schedule,
the model implements **adaptive decision rules** that expand PV and BESS capacity
**contingent on observed Effective Availability (EA)**.

### Decision Rules — Eqs. (31)–(32)

**PV expansion rule F_t** (expand when grid can absorb more):
- EA < 5% (low curtailment) → add up to **40 MW** (θ_H)
- 5% ≤ EA < 8% → add **25 MW** (θ_M)
- 8% ≤ EA < 10% → add **10 MW** (θ_L)
- EA ≥ 10% → **no expansion** (curtailment too high to justify more PV)

**BESS expansion rule H_t** (expand when curtailment is high — store excess):
- EA > 15% (high curtailment) → add up to **40 MW** (δ_H)
- 10% < EA ≤ 15% → add **20 MW** (δ_M)
- 5% ≤ EA ≤ 10% → add **10 MW** (δ_L)
- EA < 5% → **no expansion** (curtailment manageable without more storage)

Note the **inverted logic**: high EA triggers BESS expansion (store the curtailed energy)
but suppresses PV expansion (grid can't absorb more generation).

### Value of Flexibility (VoF)
> VoF = E[LCOE_rigid] − E[LCOE_flexible]

A positive VoF indicates that the flexible strategy reduces expected lifecycle costs
compared to the best fixed strategy.


In [ ]:
# =============================================================================
# STEP 3: FLEXIBLE DEPLOYMENT WITH EA-TRIGGERED DECISION RULES
# =============================================================================
# Implements the adaptive capacity expansion model of Section 2.1.3.
# Decision rules F_t (PV) and H_t (BESS) from Eqs. (26)–(27) are
# operationalised via threshold-based conditions on EA — Eqs. (31)–(32).

print("Start of Step 3B")

# --- Effective Availability (EA) projection curves ---
# Historical EA data for the Alentejo site (25-year horizon)
# EA_baseline: minimum curtailment scenario (near-zero losses)
# EA_higher: higher curtailment scenario (up to ~24% by year 20+)
years_in_operation = np.arange(1, 26)  # Years 1 through 25

EA_baseline3A = np.array([
    0.0118, 0.0043, 0.0033, 0.0089, 0.0036, 0.0034, 0.0031, 0.0030,
    0.0032, 0.0027, 0.0025, 0.0024, 0.0024, 0.0022, 0.0021, 0.0023,
    0.0018, 0.0016, 0.0015, 0.0011, 0.0010, 0.0007, 0.0007, 0.0007, 0.0007
])

EA_higher3A = np.array([
    0.1191, 0.1413, 0.1604, 0.1430, 0.1489, 0.1437, 0.1633, 0.1558,
    0.1704, 0.1731, 0.2125, 0.2427, 0.2277, 0.2038, 0.2058, 0.2200,
    0.2274, 0.2301, 0.2402, 0.2403, 0.2339, 0.2402, 0.2402, 0.2402, 0.2402
])

EA_mean3A = (EA_baseline3A + EA_higher3A) / 2  # Mode for triangular distribution

# Interpolation functions for continuous EA sampling at any year
f_baseline_step3A = interp1d(years_in_operation, EA_baseline3A, kind='linear', fill_value="extrapolate")
f_mean_step3A = interp1d(years_in_operation, EA_mean3A, kind='linear', fill_value="extrapolate")
f_higher_step3A = interp1d(years_in_operation, EA_higher3A, kind='linear', fill_value="extrapolate")


def get_EA_3A(year_of_operation):
    """
    Sample Effective Availability (EA) for a given year from a triangular distribution.
    EA represents the fraction of potential generation lost to curtailment and
    operational disruptions — Eq. (25): EA_t^s = E_t^s / E_t^{max,s}
    The triangular distribution is bounded by historical baseline (optimistic)
    and higher-curtailment (pessimistic) projections, with the mean as the mode.
    
    Returns:
        float: Sampled EA value (0 = no curtailment, 1 = total curtailment)
    """
    baseline3A = f_baseline_step3A(year_of_operation)
    higher3A = f_higher_step3A(year_of_operation)
    mean3A = f_mean_step3A(year_of_operation)
    return np.random.triangular(baseline3A, mean3A, higher3A)


### Decision Rule Functions

These functions implement the **adaptive expansion policy** — Eqs. (31) and (32).
They are called each year to determine how much PV/BESS capacity to add based on the
most recently observed EA value.

The expansion window is capped at **year 5** — beyond that, no further capacity additions
are permitted (reflecting typical project development timelines).


In [ ]:
# =============================================================================
# CONFIGURATION SPACE FOR FLEXIBLE DEPLOYMENT
# =============================================================================
storage_durations_step3A = [1]                         # [hours] Storage duration
initial_BESS_capacity3A = [0, 10, 20, 30, 40, 60]     # [MW] Starting BESS capacities to sweep
initial_PV_capacity3A = [25, 50, 75, 100]              # [MW] Starting PV capacities to sweep


def decide_capacity_addition_PV3A(current_EA3A, total_installed_PV3A, current_year, max_pv3A=100):
    """
    PV expansion decision rule F_t — Eq. (31).
    
    Logic: Install MORE PV when EA (curtailment) is LOW, meaning the grid can
    absorb additional generation. When EA is high, curtailment losses make
    further PV investment uneconomical.
    
    Thresholds map to paper notation:
        current_EA3A < τ_EA (0.05)  → θ_H = 40 MW  (high expansion)
        τ_EA ≤ EA < ρ_EA (0.08)    → θ_M = 25 MW  (medium expansion)
        ρ_EA ≤ EA < μ_EA (0.10)    → θ_L = 10 MW  (low expansion)
        EA ≥ μ_EA                   → 0 MW         (no expansion)
    
    Args:
        current_EA3A: Observed EA in the previous period
        total_installed_PV3A: Cumulative installed PV capacity [MW]
        current_year: Current simulation year
        max_pv3A: Maximum allowable PV capacity θ_max [MW]
    
    Returns:
        int: MW of PV capacity to add this year
    """
    if current_year >= 5:  # Expansion window closed after year 5 (can be modified according to the user needs)
        return 0
    remaining_capacity3A = max_pv3A - total_installed_PV3A
    
    if current_EA3A < 0.05:        # Very low curtailment → aggressive expansion
        if remaining_capacity3A >= 40:
            return 40
        elif remaining_capacity3A > 0:
            return remaining_capacity3A
        else:
            return 0
    elif 0.05 <= current_EA3A < 0.08:   # Moderate curtailment → medium expansion
        if (total_installed_PV3A + 25) <= max_pv3A:
            return 25
        else:
            return 0
    elif 0.08 <= current_EA3A < 0.10:   # High curtailment → cautious expansion
        if (total_installed_PV3A + 10) <= max_pv3A:
            return 10
        else:
            return 0
    else:                               # Very high curtailment → no expansion
        return 0


def decide_capacity_addition_BESS3A(current_EA3A, total_installed_BESS3A, current_year, max_bess3A=60):
    """
    BESS expansion decision rule H_t — Eq. (32).
    
    Logic (INVERTED vs PV): Install MORE BESS when EA (curtailment) is HIGH,
    because storage can capture curtailed energy for later dispatch (arbitrage).
    When EA is low, the grid absorbs generation directly and storage adds less value.
    
    Thresholds map to paper notation (inverted):
        current_EA3A > Ω_EA (0.15)   → δ_H = 40 MW  (high expansion)
        η_EA < EA ≤ Ω_EA (0.10-0.15) → δ_M = 20 MW  (medium expansion)
        Ω̲_EA ≤ EA ≤ η_EA (0.05-0.10) → δ_L = 10 MW  (low expansion)
        EA < Ω̲_EA (0.05)             → 0 MW         (no expansion)
    
    Args:
        current_EA3A: Observed EA in the previous period
        total_installed_BESS3A: Cumulative installed BESS capacity [MW]
        current_year: Current simulation year
        max_bess3A: Maximum allowable BESS capacity δ_max [MW]
    
    Returns:
        int: MW of BESS capacity to add this year
    """
    if current_year >= 5:  # Expansion window closed after year 5
        return 0

    if current_EA3A > 0.15:                          # Very high curtailment → aggressive BESS build
        remaining_capacity3A = max_bess3A - total_installed_BESS3A
        if remaining_capacity3A >= 40:
            return 40
        elif remaining_capacity3A > 0:
            return remaining_capacity3A
        else:
            return 0
    elif 0.1 < current_EA3A <= 0.15:                 # High curtailment → medium BESS build
        return 20 if (total_installed_BESS3A + 30) <= max_bess3A else 0
    elif 0.05 <= current_EA3A <= 0.1:                # Moderate curtailment → small BESS build
        return 10 if (total_installed_BESS3A + 10) <= max_bess3A else 0
    else:                                             # Low curtailment → no BESS needed
        return 0


def generate_dynamic_schedule_PV3A(initial_PV_capacity3A, EA_values):
    """
    Generate a path-dependent PV deployment schedule based on EA decision rules.
    
    Starting from an initial capacity, applies F_t each year to build
    the contingent capacity sequence χ — Eq. (24).
    
    Returns:
        list: Year-by-year PV capacity additions [MW]
    """
    schedule_PV3A = [initial_PV_capacity3A]
    total_installed_PV_capacity3A = initial_PV_capacity3A
    
    for year in range(1, project_lifetime):
        current_EA3A = EA_values3A[year] if year < len(EA_values3A) else EA_values3A[-1]
        capacity_PV_addition3A = decide_capacity_addition_PV3A(current_EA3A, total_installed_PV_capacity3A, year)
        schedule_PV3A.append(capacity_PV_addition3A)
        total_installed_PV_capacity3A += capacity_PV_addition3A
    
    return schedule_PV3A


def generate_dynamic_schedule_BESS3A(initial_BESS_capacity3A, EA_values3A):
    """
    Generate a path-dependent BESS deployment schedule based on EA decision rules.
    
    Starting from an initial capacity, applies H_t each year to build
    the contingent storage sequence — Eq. (24).
    
    Returns:
        list: Year-by-year BESS capacity additions [MW]
    """
    schedule_BESS3A = [initial_BESS_capacity3A]
    total_installed_BESS_capacity3A = initial_BESS_capacity3A
    
    for year in range(1, project_lifetime):
        current_EA3A = EA_values3A[year] if year < len(EA_values3A) else EA_values3A[-1]
        capacity_BESS_addition3A = decide_capacity_addition_BESS3A(current_EA3A, total_installed_BESS_capacity3A, year)
        schedule_BESS3A.append(capacity_BESS_addition3A)
        total_installed_BESS_capacity3A += capacity_BESS_addition3A
    
    return schedule_BESS3A


### Step 3 — Monte Carlo Simulation with Flexible Decision Rules

**Paper reference: Eq. (28)** — the objective function with expansion costs:

> min E[LCOE] = Σ_s p_s · Σ_t [C_t(θ,δ,φ) + C_t^{exp,s}] / (1+λ)^t ÷ Σ_t E_t(φ) / (1+λ)^t

For each Monte Carlo iteration:
1. **Sample** a 20-year EA trajectory from the triangular distribution
2. **Apply decision rules** F_t and H_t to generate path-dependent PV and BESS schedules
3. **Compute** the scenario-specific LCOE/LCOS including expansion costs Υ — Eq. (29)

The simulation sweeps over all combinations of initial PV capacity (25–100 MW)
and initial BESS capacity (0–60 MW), evaluating how different starting points
interact with the adaptive expansion rules under uncertainty.


In [ ]:
# =============================================================================
# STEP 3 MONTE CARLO SIMULATION — Flexible Deployment with Decision Rules
# =============================================================================
# For each simulation:
#   1. Sample a 20-year EA trajectory → get_EA_3A()
#   2. Apply decision rules to generate path-dependent schedules
#   3. Compute LCOE/LCOS with expansion costs — Eq. (28)
#
# The simulation sweeps over:
#   - Initial PV capacities: [25, 50, 75, 100] MW
#   - Initial BESS capacities: [0, 10, 20, 30, 40, 60] MW
#   - All module types × tracker types
#
# This produces the distributional LCOE output needed to compute the
# Value of Flexibility (VoF) = E[LCOE_rigid] - E[LCOE_flexible]

monte_carlo_iterations_step3A = 2000  #can be modified
results_step3A = []

#for degradation_rate in degradation_rate:
for simulation in range(monte_carlo_iterations_step3A):
        EA_values3A = [get_EA_3A(year) for year in range(project_lifetime)] 
        for initial_pv3A in initial_PV_capacity3A:
            schedule_PV3A = generate_dynamic_schedule_PV3A(initial_pv3A, EA_values3A)
            for initial_bess3A in initial_BESS_capacity3A:
                # Generate dynamic schedules using the EA values
                schedule_BESS3A = generate_dynamic_schedule_BESS3A(initial_bess3A, EA_values3A)
                
                for storage_hours in storage_durations_step3A:      
                    storage_hours_rounded = int(round(storage_hours))
                                    
                    for tracker, tracker_cost_dict in tracking_costs.items():
                        tracker_multiplier = tracker_multipliers[tracker]
                        
                        tracker_opex_per_kw_step3A = np.random.triangular(
                            0.8 * opex_per_kw_per_year[tracker],
                            opex_per_kw_per_year[tracker],
                            1.2 * opex_per_kw_per_year[tracker]
                        )
                        
                        for module, module_info in module_data.items():
                            # Use the dynamic schedules instead of installation_scenarios_step3
                            last_installation_year = max([idx for idx, cap in enumerate(schedule_PV3A) if cap > 0], default=0)
                            extended_project_lifetime = last_installation_year + 21
                            module_efficiency_multiplier = module_efficiency[module]
                            
                            # INITIALIZE ALL FINANCIAL TRACKERS
                            total_solar_capex = 0.0
                            total_battery_capex = 0.0
                            npv_numerator_lcoe = 0.0
                            npv_denominator_lcoe = 0.0
                            npv_numerator_lcos = 0.0
                            npv_denominator_lcos = 0.0
                            cumulative_installed_capacity_mw = 0
                            cumulative_battery_installed_capacity_mw = 0
                            active_generations = [0] * extended_project_lifetime
                            battery_generations_discharge = [0] * extended_project_lifetime
                            Salvage_total_lcoe = 0
                            Salvage_total_lcos = 0
                            max_pv_reached = 0
                            max_bess_reached = 0
                            
                            # Initialize decommissioned capacities
                            decommissioned_capacity = 0
                            decommissioned_capacity_battery = 0
                            
                            for year in range(extended_project_lifetime):
                                discount_factor = 1 / ((1 + discount_rate) ** year) if year > 0 else 1
                                
                                # CAPACITY TRACKING - Use dynamic schedules
                                installed_capacity_mw = schedule_PV3A[year] if year < len(schedule_PV3A) else 0
                                battery_installed_capacity_mw = schedule_BESS3A[year] if year < len(schedule_BESS3A) else 0
                                
                                if year >= 20:
                                    decommissioned_capacity = schedule_PV3A[year - 20] if (year - 20) < len(schedule_PV3A) else 0
                                    cumulative_installed_capacity_mw -= decommissioned_capacity
                                    decommissioned_capacity_battery = schedule_BESS3A[year - 20] if (year - 20) < len(schedule_BESS3A) else 0
                                    cumulative_battery_installed_capacity_mw -= decommissioned_capacity_battery
                                
                                cumulative_installed_capacity_mw += installed_capacity_mw
                                cumulative_battery_installed_capacity_mw += battery_installed_capacity_mw
                                
                                max_pv_reached = max(max_pv_reached, cumulative_installed_capacity_mw)
                                max_bess_reached = max(max_bess_reached, cumulative_battery_installed_capacity_mw)
                                
                                # SOLAR CAPEX (same as before)
                                if installed_capacity_mw > 0:
                                    cost_category = determine_cost_category(installed_capacity_mw)
                                    tracker_cost_per_watt_step3A = np.random.triangular(
                                        0.8 * tracker_cost_dict[cost_category],
                                        tracker_cost_dict[cost_category],
                                        1.2 * tracker_cost_dict[cost_category]
                                    )
                                    
    
                                    module_cost_per_watt_step3A = np.random.triangular(
                                        0.8 * module_info["cost_per_watt"],
                                        module_info["cost_per_watt"],
                                        1.2 * module_info["cost_per_watt"]
                                    )
                                    
                                    lf_tech = (1 - Learning_rate_technology) ** max(0, year - 1)
                                    lf_exp = (1 - Learning_rate_experience) ** max(0, year - 1)
                                    
                                    plant_capacity_w = installed_capacity_mw * 1e6
                                    module_cost = plant_capacity_w * module_cost_per_watt_step3A * lf_tech*(cumulative_installed_capacity_mw ** (1 - 1))  #last part are the economies of scale
                                    tracker_cost = plant_capacity_w * tracker_cost_per_watt_step3A * lf_exp*(cumulative_installed_capacity_mw ** (1 - 1)) #last part are the economies of scale
                                    yearly_solar_capex = module_cost + tracker_cost
                                    total_solar_capex += yearly_solar_capex * discount_factor
                                
                                # BATTERY CAPEX (same as before)
                                if battery_installed_capacity_mw < 10:
                                    battery_cost_dict = cost_dict
                                elif battery_installed_capacity_mw < 61:
                                    battery_cost_dict = cost_dict_10
                                else:
                                    battery_cost_dict = cost_dict_100

                                                                  
                                lf_BESS = (1 - learning_rate_BESS1) ** max(0, year - 1)
                                    
                                battery_capacity_kwh = battery_installed_capacity_mw * 1e3 * storage_hours_rounded
                                battery_capacity_kw = battery_installed_capacity_mw * 1e3
                                    
                                battery_cost3A = battery_capacity_kwh * (   #description of all costs associated with the BESS
                                    np.random.triangular(
                                        0.9 * battery_cost_dict[f"{storage_hours_rounded} hours"]["DC Storage Block ($/kWh)"],
                                        battery_cost_dict[f"{storage_hours_rounded} hours"]["DC Storage Block ($/kWh)"],
                                        1.1 * battery_cost_dict[f"{storage_hours_rounded} hours"]["DC Storage Block ($/kWh)"]
                                        ) +
                                        np.random.triangular(
                                        0.9 * battery_cost_dict[f"{storage_hours_rounded} hours"]["DC Storage BOS ($/kWh)"],
                                        battery_cost_dict[f"{storage_hours_rounded} hours"]["DC Storage BOS ($/kWh)"],
                                        1.1 * battery_cost_dict[f"{storage_hours_rounded} hours"]["DC Storage BOS ($/kWh)"]
                                        ) +
                                        np.random.triangular(
                                        0.9 * battery_cost_dict[f"{storage_hours_rounded} hours"]["Systems Integration ($/kWh)"],
                                        battery_cost_dict[f"{storage_hours_rounded} hours"]["Systems Integration ($/kWh)"],
                                        1.1 * battery_cost_dict[f"{storage_hours_rounded} hours"]["Systems Integration ($/kWh)"]
                                        ) +
                                        np.random.triangular(
                                        0.9 * battery_cost_dict[f"{storage_hours_rounded} hours"]["EPC ($/kWh)"],
                                        battery_cost_dict[f"{storage_hours_rounded} hours"]["EPC ($/kWh)"],
                                        1.1 * battery_cost_dict[f"{storage_hours_rounded} hours"]["EPC ($/kWh)"]
                                        ) +
                                        np.random.triangular(
                                        0.9 * battery_cost_dict[f"{storage_hours_rounded} hours"]["Project Development ($/kWh)"],
                                        battery_cost_dict[f"{storage_hours_rounded} hours"]["Project Development ($/kWh)"],
                                        1.1 * battery_cost_dict[f"{storage_hours_rounded} hours"]["Project Development ($/kWh)"]
                                        ))* Dolar * lf_BESS + battery_capacity_kw * (
                                        np.random.triangular(
                                        0.9 * battery_cost_dict[f"{storage_hours_rounded} hours"]["Power Equipment ($/kW)"],
                                        battery_cost_dict[f"{storage_hours_rounded} hours"]["Power Equipment ($/kW)"],
                                        1.1 * battery_cost_dict[f"{storage_hours_rounded} hours"]["Power Equipment ($/kW)"]
                                        )+
                                        np.random.triangular(
                                        0.9 * battery_cost_dict[f"{storage_hours_rounded} hours"]["CC ($/kW)"],
                                        battery_cost_dict[f"{storage_hours_rounded} hours"]["CC ($/kW)"],
                                        1.1 * battery_cost_dict[f"{storage_hours_rounded} hours"]["CC ($/kW)"]
                                        )+
                                        np.random.triangular(
                                        0.9 * battery_cost_dict[f"{storage_hours_rounded} hours"]["Grid Integration ($/kW)"],
                                        battery_cost_dict[f"{storage_hours_rounded} hours"]["Grid Integration ($/kW)"],
                                        1.1 * battery_cost_dict[f"{storage_hours_rounded} hours"]["Grid Integration ($/kW)"]
                                        )) * Dolar*lf_BESS
                                 

                                battery_cost_scale= battery_cost3A*(cumulative_battery_installed_capacity_mw** (1 - 1)) if cumulative_battery_installed_capacity_mw > 0 else 0
                                    
                                total_battery_capex += battery_cost_scale * discount_factor
                                
                                # ENERGY GENERATION CALCULATIONS (same as before)
                                raw_generations_for_year = 0
                                if year>0:  
                                    for previous_year in range(max(0, year - 19), year + 1):
                                        active_capacity_mw = schedule_PV3A[previous_year] if previous_year < len(schedule_PV3A) else 0
                                        degradation_factor = (1 - degradation_rate) ** (year - previous_year)
                                        raw_generations_for_year += (
                                            P_profiles[min(year, len(P_profiles) - 1)]
                                            * active_capacity_mw / Power_capacity
                                            * tracker_multiplier
                                            * module_efficiency_multiplier
                                            * degradation_factor
                                        )
                                    
                                        EA_pct3A = get_EA_3A(min(year + 1,25))
                                        active_generation_for_year = raw_generations_for_year * (1 - EA_pct3A)
                                        
                                    # Battery Discharge (for LCOS)
                                    cycles_per_year = 4500 / 20  # 4500 cycles over 20 years
        
                                    # Battery discharge calculations
                                    battery_generation_for_year = 0
                                    
                                    for previous_year in range(max(0, year - 19), year + 1):
                                        battery_active_capacity_mw = schedule_BESS3A[previous_year] if previous_year < len(schedule_BESS3A) else 0
                                        degradation_battery_factor = (1 - Battery_degradation1) ** (year - previous_year)
                                        battery_generation_for_year += (
                                            battery_active_capacity_mw
                                            * storage_hours_rounded
                                            * battery_efficiency
                                            * DoD
                                            * degradation_battery_factor
                                            * cycles_per_year
                                        )
    
                                    npv_numerator_lcoe += (
                                    tracker_opex_per_kw_step3A * cumulative_installed_capacity_mw * 1e3 * discount_factor +
                                    Land_Renting / 20 * installed_capacity_mw * discount_factor +
                                    np.random.triangular(
                                        0.9 * battery_cost_dict[f"{storage_hours_rounded} hours"]["Operation Cost ($/kW-year)"],
                                        battery_cost_dict[f"{storage_hours_rounded} hours"]["Operation Cost ($/kW-year)"],
                                        1.1 * battery_cost_dict[f"{storage_hours_rounded} hours"]["Operation Cost ($/kW-year)"]
                                        ) * battery_capacity_kw * discount_factor)
                                
                                    # LCOS Numerator (costs)
                                    npv_numerator_lcos += np.random.triangular(
                                        0.9 * battery_cost_dict[f"{storage_hours_rounded} hours"]["Operation Cost ($/kW-year)"],
                                        battery_cost_dict[f"{storage_hours_rounded} hours"]["Operation Cost ($/kW-year)"],
                                        1.1 * battery_cost_dict[f"{storage_hours_rounded} hours"]["Operation Cost ($/kW-year)"]
                                        ) * battery_capacity_kw * discount_factor
    
                                    active_generations[year] = active_generation_for_year 
                                    battery_generations_discharge[year] = battery_generation_for_year
    
                                
                                    # LCOE Denominator (energy generation)
                                    npv_denominator_lcoe += (active_generations[year] + battery_generations_discharge[year])* discount_factor  #active generations [year] before
               
                                    # LCOS Denominator (energy discharged)
                                    npv_denominator_lcos += battery_generations_discharge[year] * discount_factor
            
                                
                                if decommissioned_capacity > 0:
                                    salvage_year = Salvage * decommissioned_capacity * discount_factor
                                    Salvage_total_lcoe += salvage_year
                                if decommissioned_capacity_battery > 0:
                                    salvage_year_battery = np.random.triangular(
                                        0.9 * Salvage_battery,
                                        Salvage_battery,
                                        1.1 * Salvage_battery) * 1e3 * storage_hours_rounded * decommissioned_capacity_battery * discount_factor
                                    Salvage_total_lcos += salvage_year_battery
    
                            salvage_total = Salvage_total_lcoe + Salvage_total_lcos
                            salvage_total_battery = Salvage_total_lcos
    
    
                            
                            # FINAL METRICS CALCULATION
                            lcoe3A = (total_solar_capex + total_battery_capex + npv_numerator_lcoe+salvage_total) / npv_denominator_lcoe if npv_denominator_lcoe != 0 else float("inf")
                            lcos3A = (total_battery_capex + npv_numerator_lcos+salvage_total_battery) / npv_denominator_lcos if npv_denominator_lcos != 0 else float("inf")
            
                            total_capex3A = total_solar_capex + total_battery_capex
                            # RESULTS APPEND
                            results_step3A.append({
                                "Initial PV Capacity": initial_pv3A,
                                "Initial BESS Capacity": initial_bess3A,
                                "Tracker Type": tracker,
                                "Module Type": module,
                                "Storage Hours": storage_hours_rounded,
                                "Degradation Rate": degradation_rate,
                                "Peak PV Capacity (MW)": max_pv_reached,
                                "Peak BESS Capacity (MW)": max_bess_reached,
                                "BESS Capacity (MWh)": cumulative_battery_installed_capacity_mw * storage_hours_rounded,
                                "Solar CAPEX (EUR)": total_solar_capex,
                                "Battery CAPEX (EUR)": total_battery_capex,
                                "LCOE (EUR/MWh)": lcoe3A,
                                "LCOS (EUR/MWh)": lcos3A,
                                "Total CAPEX (EUR)": total_capex3A,
                                "Simulation": simulation
                            })
        # Convert to DataFrame
results_step3A_df = pd.DataFrame(results_step3A)

# Filter for IBC module and Fixed tracker
ibc_fixed_df = results_step3A_df[
    (results_step3A_df['Module Type'] == 'IBC') &
    (results_step3A_df['Tracker Type'] == 'Fixed')
]

# Select identifying columns and the LCOE/LCOS values
ibc_fixed_export = ibc_fixed_df[[
    "Initial PV Capacity",
    "Initial BESS Capacity",
    "Storage Hours",
    "Degradation Rate",
    "Simulation",
    "LCOE (EUR/MWh)",
    "LCOS (EUR/MWh)"
]]

# Export to CSV
ibc_fixed_export.to_csv("lcoe_lcos_ibc_fixed_montecarlo.csv", index=False)
#print("Exported LCOE and LCOS for IBC, Fixed Axis configurations to lcoe_lcos_ibc_fixed_montecarlo.csv")

# Loop over each unique discount rate
for dr in sorted(results_step3A_df['Degradation Rate'].unique()):
    df_dr = results_step3A_df[
        (results_step3A_df['Degradation Rate'] == dr) &
        (results_step3A_df['Module Type'] == 'IBC') & 
        (results_step3A_df['Tracker Type'] == 'Fixed')
    ]

    stats_df = df_dr.groupby(['Initial PV Capacity', 'Initial BESS Capacity']).agg({
        'LCOE (EUR/MWh)': [lambda x: x.quantile(0.95), lambda x: x.quantile(0.05)],
        'Total CAPEX (EUR)': 'mean'
    }).reset_index()

    stats_df.columns = [
        'Initial PV Capacity (MW)',
        'Initial BESS Capacity (MW)',
        'P95 LCOE (EUR/MWh)',
        'P5 LCOE (EUR/MWh)',
        'Avg CAPEX (EUR)'
    ]

    # Calculate P50 (median) LCOE for each (PV, BESS) configuration
    lcoe_p50 = (
    df_dr.groupby(['Initial PV Capacity', 'Initial BESS Capacity'])['LCOE (EUR/MWh)']
    .median()
    .reset_index()
        )
    
    # Pivot for heatmap
    heatmap_data = lcoe_p50.pivot(
        index='Initial PV Capacity', 
        columns='Initial BESS Capacity', 
        values='LCOE (EUR/MWh)'
    ).iloc[::-1]  # Reverse for visual convention

    plt.figure(figsize=(10, 6))
    ax = sns.heatmap(
        heatmap_data,
        annot=True,
        fmt=".1f",
        cmap="crest",
        linewidths=0.5,
        cbar_kws={'label': 'P50 LCOE (€/MWh)'},
        annot_kws={'size': 14}
    )
    plt.title(f"P50 LCOE for Fixed-Tracker IBC Modules (Degradation Rate PV = {dr:.2%})", fontsize=18)
    plt.xlabel("Initial BESS Capacity (MW/MWh)", fontsize=16)
    plt.ylabel("Initial PV Capacity (MW)", fontsize=16)
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)
    
    # Adjust colorbar tick size
    cbar = plt.gcf().axes[-1]
    cbar.tick_params(labelsize=14)
    
    plt.tight_layout()
    plt.savefig("heatmap_p50_lcoe_fixed_ibc_new_colors.png", dpi=900, bbox_inches='tight')
    plt.show()
    
